# V9.2 — Partie 2 V5 : Normalisation, DQ, CarthagoDom et validation manuelle unique

Cette V5 conserve la logique réglementaire de la V4 et ajoute :

- une normalisation métier dédiée de `CTR_REFERENCE_DOMICILIATION` ;
- l'extraction robuste de `SIEGE_RACINE` depuis `DOM_COMPTE_LOCAL` par regex `07310 + 6 chiffres` ;
- le rapprochement avec `Dossier de domiciliation.xlsx` (DOM / PREDOM) ;
- la comparaison exacte `SIEGE_RACINE == Client` ;
- une similarité **informationnelle** entre `CTR_NOM_PRENOM_TRAVAILLEUR` et `Nom complet/Raison sociale` ;
- une proposition de numéro de domiciliation CarthagoDom lorsque le rapprochement est non ambigu ;
- une ligne `NUMERO_DOMICILIATION_VALIDE` dans la même feuille `A_VALIDER_MANUELLEMENT` pour chaque dossier ;
- aucune utilisation automatique de la date de signature / date CTS comme date d'effet de l'augmentation.

Le RAW V9.2 et son schéma de 99 champs restent immuables. Les champs Carthago et les informations de rapprochement sont des données dérivées de Partie 2.


## 1. Imports et configuration


In [ ]:
import hashlib
import json
import math
import re
from collections import defaultdict
from datetime import datetime
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd

SCHEMA_VERSION = 'DOM_EXTRACTION_V1'
EXPECTED_FIELD_SCHEMA_HASH = 'da633243929ec467ed246f823571e23de5500442a8510a2a341a78b152aa5a5e'
POSTPROCESS_VERSION = 'DOM_V9_2_PART2_DQ_CONFIDENCE_V5'
NORMALIZATION_VERSION = 'NORM_V9_2_CONSERVATIVE_V3_DOM_REF'
MAX_DOSSIERS = None   # mettre 10 pour un test

OUTPUT_ROOT = Path('/mnt/data/domiciliations_v9')
RAW_JSON_DIR = OUTPUT_ROOT / '01_extraction_raw' / 'json_dossiers'
POST_ROOT = OUTPUT_ROOT / '02_postprocessing'
PROCESSED_JSON_DIR = POST_ROOT / 'processed_json'
VALIDATION_XLSX = POST_ROOT / 'validation_domiciliations_v9.xlsx'
DOSSIERS_CSV = POST_ROOT / 'dossiers_a_valider.csv'
FIELDS_CSV = POST_ROOT / 'champs_detail.csv'
ANOMALIES_CSV = POST_ROOT / 'anomalies.csv'
RETRY_CSV = POST_ROOT / 'vlm_retry_requests.csv'
DQ_SUMMARY_CSV = POST_ROOT / 'dq_summary.csv'
FIELD_CONFIDENCE_CSV = POST_ROOT / 'field_confidence.csv'

# Référentiel CarthagoDom — V5
# Le fichier peut être placé dans /mnt/data ou le chemin peut être modifié ici.
CARTHAGO_XLSX = Path('/mnt/data/Dossier de domiciliation.xlsx')
CARTHAGO_CLIENT_COL = 'Client'
CARTHAGO_NAME_COL = 'Nom complet/Raison sociale'
# Laisser None pour auto-détection prudente ; renseigner le nom exact si nécessaire.
CARTHAGO_DOM_REF_COL = None
CARTHAGO_TYPE_COL = None

CARTHAGO_MATCH_CSV = POST_ROOT / 'carthago_domiciliation_match.csv'

POST_ROOT.mkdir(parents=True,exist_ok=True)
PROCESSED_JSON_DIR.mkdir(parents=True,exist_ok=True)

print('RAW :',RAW_JSON_DIR)
print('Post:',POST_ROOT)


## 2. Schéma — mêmes 99 champs V8.1 / V9


In [ ]:
FIELD_SCHEMA = {'ENGAGEMENT_DOMICILIATION': ['DOM_NOM_RAISON_SOCIAL_CLIENT', 'DOM_COMPTE_LOCAL', 'DOM_ADRESSE_CLIENT', 'DOM_AGENCE_DOMICILIATAIRE', 'DOM_NUMERO_CONTRAT', 'DOM_DUREE_CONTRAT_MOIS', 'DOM_DATE_DEBUT_CONTRAT', 'DOM_DATE_FIN_CONTRAT', 'DOM_NOM_RAISON_SOCIAL_EMPLOYEUR', 'DOM_ADRESSE_EMPLOYEUR', 'DOM_SALAIRE_NET_MENSUEL', 'DOM_PART_TRANSFERABLE', 'DOM_TAUX_TRANSFERABLE', 'DOM_MONTANT_TOTAL_DOMICILIE', 'DOM_DATE_SIGNATURE'], 'CONTRAT_TRAVAIL': ['CTR_REFERENCE_DOCUMENT', 'CTR_TYPE', 'CTR_EMPLOYEUR', 'CTR_ACTIVITE_EMPLOYEUR', 'CTR_DUREE_MOIS', 'CTR_DATE_DEBUT_CONTRAT', 'CTR_POSTE', 'CTR_NOM_PRENOM_TRAVAILLEUR', 'CTR_PERE_NOM_PRENOM', 'CTR_MERE_NOM_PRENOM', 'CTR_NATIONALITE', 'CTR_DATE_NAISSANCE', 'CTR_LIEU_PAYS_NAISSANCE', 'CTR_ADRESSE_ALGERIE', 'CTR_QUALIFICATION', 'CTR_NUMERO_PERMIS_TRAVAIL', 'CTR_DATE_DELIVRANCE_PERMIS', 'CTR_DATE_DEBUT_VALIDITE_PERMIS', 'CTR_DATE_FIN_VALIDITE_PERMIS', 'CTR_SALAIRE_BRUT', 'CTR_SALAIRE_NET', 'CTR_AFFILIATION_SS', 'CTR_NUMERO_EMPLOYEUR', 'CTR_DATE_SIGNATURE', 'CTR_REFERENCE_DOMICILIATION', 'CTR_SIGNATURE_TRAVAILLEUR_PRESENTE', 'CTR_SIGNATURE_EMPLOYEUR_PRESENTE', 'CTR_CACHET_EMPLOYEUR_PRESENT'], 'CONTRAT_SPECIFIQUE': ['CTS_REFERENCE_DOCUMENT', 'CTS_SAP_ID', 'CTS_EMPLOYEUR', 'CTS_ACTIVITE_EMPLOYEUR', 'CTS_DUREE_MOIS', 'CTS_DATE_DEBUT_CONTRAT', 'CTS_POSTE', 'CTS_NOM_PRENOM_TRAVAILLEUR', 'CTS_PERE_NOM_PRENOM', 'CTS_MERE_NOM_PRENOM', 'CTS_NATIONALITE', 'CTS_DATE_NAISSANCE', 'CTS_LIEU_PAYS_NAISSANCE', 'CTS_ADRESSE_ALGERIE', 'CTS_QUALIFICATION', 'CTS_NUMERO_PERMIS_TRAVAIL', 'CTS_DATE_DELIVRANCE_PERMIS', 'CTS_DATE_DEBUT_VALIDITE_PERMIS', 'CTS_DATE_FIN_VALIDITE_PERMIS', 'CTS_LIGNE_SALAIRE_BRUTE', 'CTS_SALAIRE_NET', 'CTS_SALAIRE_NET_ANCIEN', 'CTS_MENTION_AU_LIEU_DE_PRESENTE', 'CTS_PART_TRANSFERABLE', 'CTS_PART_PAYABLE_DZD', 'CTS_NUMERO_SS_PAYS_ORIGINE', 'CTS_NUMERO_SS_ALGERIE', 'CTS_DATE_DOCUMENT', 'CTS_SIGNATURE_TRAVAILLEUR_PRESENTE', 'CTS_SIGNATURE_EMPLOYEUR_PRESENTE', 'CTS_CACHET_EMPLOYEUR_PRESENT', 'CTS_VISA_INSPECTION_TRAVAIL_PRESENT'], 'TITRE_TRAVAIL': ['TTR_NUMERO_PERMIS', 'TTR_NUMERO_MANUSCRIT', 'TTR_POSTE', 'TTR_DUREE', 'TTR_DATE_DEBUT', 'TTR_DATE_FIN', 'TTR_LIEU_TRAVAIL', 'TTR_EMPLOYEUR', 'TTR_ADRESSE_EMPLOYEUR', 'TTR_FAIT_A', 'TTR_DATE_DELIVRANCE', 'TTR_NOM', 'TTR_PRENOM', 'TTR_DATE_NAISSANCE', 'TTR_LIEU_NAISSANCE', 'TTR_PAYS', 'TTR_NATIONALITE', 'TTR_QUALIFICATION', 'TTR_DATE_ENTREE_ALGERIE', 'TTR_PHOTO_PRESENTE', 'TTR_CACHET_PRESENT'], 'PERMIS_TRAVAIL_COUVERTURE': ['PTR_NUMERO_SERIE', 'PTR_WILAYA', 'PTR_CACHET_DIRECTION_EMPLOI_PRESENT']}

ALL_FIELDS=[f for fields in FIELD_SCHEMA.values() for f in fields]
assert len(ALL_FIELDS)==99 and len(set(ALL_FIELDS))==99
schema_hash=hashlib.sha256(json.dumps(FIELD_SCHEMA,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
assert schema_hash==EXPECTED_FIELD_SCHEMA_HASH
print('✅ Schéma 99 champs vérifié |',schema_hash[:16]+'…')


## 3. Typage des champs — configuration évolutive


In [ ]:
AMOUNT_FIELDS={
    'DOM_SALAIRE_NET_MENSUEL','DOM_PART_TRANSFERABLE','DOM_MONTANT_TOTAL_DOMICILIE',
    'CTR_SALAIRE_BRUT','CTR_SALAIRE_NET','CTS_SALAIRE_NET','CTS_SALAIRE_NET_ANCIEN',
    'CTS_PART_TRANSFERABLE','CTS_PART_PAYABLE_DZD',
}
PERCENT_FIELDS={'DOM_TAUX_TRANSFERABLE'}
INTEGER_FIELDS={'DOM_DUREE_CONTRAT_MOIS','CTR_DUREE_MOIS','CTS_DUREE_MOIS'}
BOOLEAN_FIELDS={
    'CTR_SIGNATURE_TRAVAILLEUR_PRESENTE','CTR_SIGNATURE_EMPLOYEUR_PRESENTE','CTR_CACHET_EMPLOYEUR_PRESENT',
    'CTS_MENTION_AU_LIEU_DE_PRESENTE','CTS_SIGNATURE_TRAVAILLEUR_PRESENTE','CTS_SIGNATURE_EMPLOYEUR_PRESENTE',
    'CTS_CACHET_EMPLOYEUR_PRESENT','CTS_VISA_INSPECTION_TRAVAIL_PRESENT',
    'TTR_PHOTO_PRESENTE','TTR_CACHET_PRESENT','PTR_CACHET_DIRECTION_EMPLOI_PRESENT',
}
DATE_FIELDS={f for f in ALL_FIELDS if '_DATE_' in f or f.startswith('TTR_DATE_') or f in {'DOM_DATE_SIGNATURE','CTS_DATE_DOCUMENT'}}
REFERENCE_FIELDS={
    'DOM_COMPTE_LOCAL','DOM_NUMERO_CONTRAT','CTR_REFERENCE_DOCUMENT','CTR_NUMERO_PERMIS_TRAVAIL',
    'CTR_REFERENCE_DOMICILIATION','CTS_REFERENCE_DOCUMENT','CTS_SAP_ID','CTS_NUMERO_PERMIS_TRAVAIL',
    'TTR_NUMERO_PERMIS','TTR_NUMERO_MANUSCRIT','PTR_NUMERO_SERIE',
}

FIELD_TYPES={}
for f in ALL_FIELDS:
    if f in AMOUNT_FIELDS: FIELD_TYPES[f]='amount'
    elif f in PERCENT_FIELDS: FIELD_TYPES[f]='percentage'
    elif f in INTEGER_FIELDS: FIELD_TYPES[f]='integer'
    elif f in BOOLEAN_FIELDS: FIELD_TYPES[f]='boolean'
    elif f in DATE_FIELDS: FIELD_TYPES[f]='date'
    elif f in REFERENCE_FIELDS: FIELD_TYPES[f]='reference'
    else: FIELD_TYPES[f]='text'

print(pd.Series(FIELD_TYPES).value_counts())


## 4. Normalisation traçable


In [ ]:
NULL_TEXTS={'','NULL','NONE','N/A','NA','NÉANT','NEANT','ILLISIBLE','NON LISIBLE'}

def result(raw,normalized,status,rule,message=None):
    return {'raw':raw,'normalized':normalized,'status':status,'rule':rule,
            'changed':(normalized != raw),'message':message}


def normalize_amount_trace(raw):
    """
    Normalisation conservatrice des montants.
    Principe réglementaire :
      - ne jamais modifier/inventer un chiffre ;
      - AUTO_OK uniquement si la structure des séparateurs est déterministe ;
      - REVIEW dès qu'une interprétation économique reste plausible.
    """
    if raw is None:
        return result(raw,None,'MISSING','AMOUNT_NULL')

    if isinstance(raw,(int,float,Decimal)) and not isinstance(raw,bool):
        try:
            return result(raw,round(float(raw),2),'RAW_OK','AMOUNT_NUMERIC')
        except Exception:
            pass

    s=str(raw).replace('\u00a0',' ').strip()
    if s.upper() in NULL_TEXTS:
        return result(raw,None,'MISSING','AMOUNT_NULL_TEXT')

    # Retirer seulement les libellés de devise connus.
    t=s.upper().replace('DZD','').replace('DA','').replace('EUR','').replace('€','').strip()
    neg=t.startswith('-')
    t=t.lstrip('+-').strip()

    # Caractères non autorisés : aucune correction OCR de type O->0 / B->8.
    if not re.fullmatch(r"[0-9., '\u2019]+", t or ''):
        return result(raw,None,'REVIEW','AMOUNT_NON_NUMERIC',
                      'caractère non numérique ambigu')

    # Les séparateurs espace/apostrophe sont acceptés comme milliers
    # seulement si leurs groupes sont structurellement cohérents.
    if re.search(r"[ '\u2019]", t):
        chunks=[x for x in re.split(r"[ '\u2019]+",t) if x]
        # Cas "454 835,67" / "454 835.67" : groupes milliers + décimales explicites.
        if len(chunks) >= 2:
            last=chunks[-1]
            # Si le dernier bloc isolé contient 1 ou 2 chiffres sans . ou ,
            # ex. "454 835 67", on refuse de deviner qu'il s'agit de décimales.
            if re.fullmatch(r'\d{1,2}', last):
                return result(raw,None,'REVIEW','AMOUNT_SPACE_DECIMAL_AMBIGUOUS',
                              'dernier groupe espace/apostrophe ambigu')
            # Tous les groupes intermédiaires doivent être des milliers.
            for ch in chunks[1:-1]:
                if not re.fullmatch(r'\d{3}', ch):
                    return result(raw,None,'REVIEW','AMOUNT_SPACE_GROUPING_AMBIGUOUS')
        t=re.sub(r"[ '\u2019]",'',t)

    if not re.fullmatch(r'[0-9.,]+',t or ''):
        return result(raw,None,'REVIEW','AMOUNT_PARSE_FAILED')

    dots=t.count('.')
    commas=t.count(',')
    dec_sep=None
    rule=''

    if dots and commas:
        last_dot=t.rfind('.')
        last_comma=t.rfind(',')
        candidate='.' if last_dot>last_comma else ','
        tail=t.split(candidate)[-1]
        if len(tail)==2:
            dec_sep=candidate
            rule='AMOUNT_MIXED_LAST_2_DECIMALS'
        elif len(tail)==1:
            dec_sep=candidate
            rule='AMOUNT_MIXED_LAST_1_DECIMAL_PAD_ZERO'
        else:
            return result(raw,None,'REVIEW','AMOUNT_MIXED_AMBIGUOUS')

    elif dots>1 or commas>1:
        sep='.' if dots else ','
        groups=t.split(sep)
        tail=groups[-1]

        # Exemple validé : 454.835.67 -> 454835.67
        if len(tail)==2 and all(g.isdigit() for g in groups):
            dec_sep=sep
            rule='AMOUNT_MULTI_GROUP_FINAL_2_DECIMALS'
        elif len(tail)==1 and all(g.isdigit() for g in groups) and all(len(g)==3 for g in groups[1:-1]):
            dec_sep=sep
            rule='AMOUNT_MULTI_GROUP_FINAL_1_DECIMAL_PAD_ZERO'
        elif all(len(g)==3 for g in groups[1:]):
            dec_sep=None
            rule='AMOUNT_MULTI_THOUSANDS'
        else:
            return result(raw,None,'REVIEW','AMOUNT_MULTI_AMBIGUOUS')

    elif dots==1 or commas==1:
        sep='.' if dots else ','
        left,right=t.split(sep)
        if len(right)==2:
            dec_sep=sep
            rule='AMOUNT_SINGLE_2_DECIMALS'
        elif len(right)==1:
            # Le séparateur décimal est explicite : 47719857,6 = 47719857.60.
            # Aucun chiffre économique n'est inventé ; le zéro final est
            # uniquement une représentation à 2 décimales.
            dec_sep=sep
            rule='AMOUNT_SINGLE_1_DECIMAL_PAD_ZERO'
        elif len(right)==3:
            return result(raw,None,'REVIEW','AMOUNT_SINGLE_3DIGITS_AMBIGUOUS')
        else:
            return result(raw,None,'REVIEW','AMOUNT_SINGLE_AMBIGUOUS')
    else:
        rule='AMOUNT_INTEGER'

    if dec_sep:
        pos=t.rfind(dec_sep)
        int_part=re.sub(r'[.,]','',t[:pos])
        dec=t[pos+1:]
        canonical=int_part+'.'+dec
    else:
        canonical=re.sub(r'[.,]','',t)

    if neg:
        canonical='-'+canonical

    try:
        val=round(float(Decimal(canonical)),2)
    except (InvalidOperation,ValueError):
        return result(raw,None,'REVIEW','AMOUNT_PARSE_FAILED')

    return result(raw,val,'AUTO_OK' if str(raw)!=str(val) else 'RAW_OK',rule)


DATE_FORMATS=['%d/%m/%Y','%d-%m-%Y','%d.%m.%Y','%Y-%m-%d','%Y/%m/%d','%Y.%m.%d']
def normalize_date_trace(raw):
    if raw is None: return result(raw,None,'MISSING','DATE_NULL')
    s=str(raw).replace('\u00a0',' ').strip()
    if s.upper() in NULL_TEXTS: return result(raw,None,'MISSING','DATE_NULL_TEXT')
    s=re.sub(r'\s+','',s)
    for fmt in DATE_FORMATS:
        try:
            dt=datetime.strptime(s,fmt)
            out=dt.strftime('%d/%m/%Y')
            return result(raw,out,'RAW_OK' if s==out else 'AUTO_OK','DATE_'+fmt.replace('%',''))
        except ValueError:
            pass
    return result(raw,None,'REVIEW','DATE_UNPARSEABLE')


def normalize_integer_trace(raw):
    if raw is None: return result(raw,None,'MISSING','INTEGER_NULL')
    if isinstance(raw,int) and not isinstance(raw,bool): return result(raw,raw,'RAW_OK','INTEGER_NUMERIC')
    s=str(raw).strip(); m=re.fullmatch(r'\s*(\d+)\s*(?:mois)?\s*',s,flags=re.I)
    if not m: return result(raw,None,'REVIEW','INTEGER_AMBIGUOUS')
    v=int(m.group(1)); return result(raw,v,'RAW_OK' if str(v)==s else 'AUTO_OK','INTEGER_EXTRACT')


def normalize_percentage_trace(raw):
    if raw is None: return result(raw,None,'MISSING','PERCENT_NULL')
    s=str(raw).strip().replace('\u00a0',' ')
    has_pct='%' in s
    s=s.replace('%','').replace(' ','').replace(',','.')
    if not re.fullmatch(r'[+-]?\d+(?:\.\d+)?',s): return result(raw,None,'REVIEW','PERCENT_AMBIGUOUS')
    v=float(s)
    if not has_pct and 0 < v <= 1:
        v*=100; rule='PERCENT_FRACTION_TO_PERCENT'
    else: rule='PERCENT_DIRECT'
    if not (0 <= v <= 100): return result(raw,None,'REVIEW','PERCENT_OUT_OF_RANGE')
    v=round(v,2); return result(raw,v,'AUTO_OK' if str(raw).strip()!=str(v) else 'RAW_OK',rule)


def normalize_boolean_trace(raw):
    if raw is None: return result(raw,None,'MISSING','BOOL_NULL')
    if isinstance(raw,bool): return result(raw,raw,'RAW_OK','BOOL_NATIVE')
    s=str(raw).strip().upper()
    if s in {'TRUE','VRAI','OUI','YES','1'}: return result(raw,True,'AUTO_OK','BOOL_TRUE_TEXT')
    if s in {'FALSE','FAUX','NON','NO','0'}: return result(raw,False,'AUTO_OK','BOOL_FALSE_TEXT')
    return result(raw,None,'REVIEW','BOOL_AMBIGUOUS')



# ---------------------------------------------------------------------
# V5 — Normalisation métier de CTR_REFERENCE_DOMICILIATION
# Format canonique attendu : PREFIXE|AAAA.T|40|SEQUENCE|DZD
# Exemple : 271901|2026.1|40|001345|DZD
# ---------------------------------------------------------------------
PIPE_TRANSLATION = str.maketrans({
    '¦': '|',
    '│': '|',
    '｜': '|',
})


def normalize_ctr_reference_domiciliation_trace(raw):
    if raw is None:
        return result(raw, None, 'MISSING', 'CTR_DOM_REF_NULL')

    s = str(raw).replace('\u00a0', ' ').strip()
    if s.upper() in NULL_TEXTS:
        return result(raw, None, 'MISSING', 'CTR_DOM_REF_NULL_TEXT')

    # Seules les variantes typographiques sûres du séparateur vertical
    # sont ramenées vers '|'. On ne transforme jamais '/' ou '-' en '|'.
    t = s.translate(PIPE_TRANSLATION)
    t = re.sub(r'\s*\|\s*', '|', t)
    parts = [x.strip() for x in t.split('|')]

    if len(parts) != 5:
        return result(
            raw, None, 'REVIEW', 'CTR_DOM_REF_STRUCTURE_EXPECTED_5_BLOCKS',
            'format attendu PREFIXE|AAAA.T|40|SEQUENCE|DZD'
        )

    prefix, periode, operation, sequence, devise = parts

    if not re.fullmatch(r'\d{6}', prefix):
        return result(raw, None, 'REVIEW', 'CTR_DOM_REF_PREFIX_INVALID',
                      'préfixe attendu sur 6 chiffres; aucune correction OCR automatique')

    # Accepte 2026.1, 2026.01, 2026.T1, 2026.T01 -> canonique 2026.1
    m = re.fullmatch(r'(\d{4})\.(?:T)?0?([1-4])', periode, flags=re.I)
    if not m:
        return result(raw, None, 'REVIEW', 'CTR_DOM_REF_YEAR_QUARTER_INVALID',
                      'période attendue AAAA.1 à AAAA.4 ou AAAA.T1 à AAAA.T4')
    year, quarter = m.group(1), m.group(2)

    # Transferts de salaire : code opération 40. Pas de correction 4O -> 40.
    if operation != '40':
        return result(raw, None, 'REVIEW', 'CTR_DOM_REF_OPERATION_NOT_40',
                      'code transfert salaire attendu = 40')

    # Le numéro séquentiel reste du texte afin de conserver les zéros initiaux.
    if not re.fullmatch(r'\d+', sequence):
        return result(raw, None, 'REVIEW', 'CTR_DOM_REF_SEQUENCE_INVALID',
                      'séquence non numérique; zéros initiaux à conserver')

    devise_up = devise.upper()
    if devise_up != 'DZD':
        return result(raw, None, 'REVIEW', 'CTR_DOM_REF_CURRENCY_NOT_DZD',
                      'transfert salaire Algérie -> étranger attendu en DZD')

    canonical = f'{prefix}|{year}.{int(quarter)}|40|{sequence}|DZD'
    status = 'RAW_OK' if canonical == s else 'AUTO_OK'
    out = result(raw, canonical, status, 'CTR_DOM_REF_STRUCTURED_CANONICAL')
    out['components'] = {
        'prefix': prefix,
        'year': int(year),
        'quarter': int(quarter),
        'operation': '40',
        'sequence': sequence,
        'currency': 'DZD',
    }
    return out

def normalize_reference_trace(raw):
    if raw is None: return result(raw,None,'MISSING','REF_NULL')
    s=str(raw).replace('\u00a0',' ').strip()
    if s.upper() in NULL_TEXTS: return result(raw,None,'MISSING','REF_NULL_TEXT')
    # Conservateur : espaces périphériques et autour de / - uniquement.
    out=re.sub(r'\s*([/\-])\s*',r'\1',re.sub(r'\s+',' ',s)).strip()
    return result(raw,out,'AUTO_OK' if out!=s else 'RAW_OK','REF_SPACING_ONLY')


def normalize_text_trace(raw):
    if raw is None: return result(raw,None,'MISSING','TEXT_NULL')
    s=str(raw).replace('\u00a0',' ').strip()
    if s.upper() in NULL_TEXTS: return result(raw,None,'MISSING','TEXT_NULL_TEXT')
    # Pas de correction orthographique / casse.
    return result(raw,s,'AUTO_OK' if s!=raw else 'RAW_OK','TEXT_TRIM_ONLY')


def normalize_field_trace(field,raw):
    if field == 'CTR_REFERENCE_DOMICILIATION':
        return normalize_ctr_reference_domiciliation_trace(raw)
    typ=FIELD_TYPES.get(field,'text')
    return {
        'amount':normalize_amount_trace,
        'date':normalize_date_trace,
        'integer':normalize_integer_trace,
        'percentage':normalize_percentage_trace,
        'boolean':normalize_boolean_trace,
        'reference':normalize_reference_trace,
        'text':normalize_text_trace,
    }[typ](raw)

# Tests de non-régression des montants.
assert normalize_amount_trace('454.835.67')['normalized']==454835.67
assert normalize_amount_trace('23.340.43')['normalized']==23340.43
assert normalize_amount_trace('23,340.43')['normalized']==23340.43
assert normalize_amount_trace('23.340,43')['normalized']==23340.43
assert normalize_amount_trace('23 340,43')['normalized']==23340.43
assert normalize_amount_trace('23,340')['status']=='REVIEW'
assert normalize_amount_trace('454 835 67')['status']=='REVIEW'
assert normalize_amount_trace('454.835.6')['normalized']==454835.60
assert normalize_amount_trace('47 719 857,6')['normalized']==47719857.60
assert normalize_amount_trace('47.719.857,6')['normalized']==47719857.60
assert normalize_amount_trace('454 835 67')['status']=='REVIEW'
assert normalize_amount_trace('454.83O.67')['status']=='REVIEW'
print('✅ Montants testés : 454.835.67 -> 454835.67 | 47 719 857,6 -> 47719857.60')


# Tests V5 — référence domiciliation.
assert normalize_ctr_reference_domiciliation_trace('271901|2026.1|40|001345|DZD')['normalized'] == '271901|2026.1|40|001345|DZD'
assert normalize_ctr_reference_domiciliation_trace('271901 | 2026.T1 | 40 | 001345 | dzd')['normalized'] == '271901|2026.1|40|001345|DZD'
assert normalize_ctr_reference_domiciliation_trace('271901|2026.5|40|001345|DZD')['status'] == 'REVIEW'
assert normalize_ctr_reference_domiciliation_trace('271901|2026.1|4O|001345|DZD')['status'] == 'REVIEW'
assert normalize_ctr_reference_domiciliation_trace('271901|2026.1|40|001345|EUR')['status'] == 'REVIEW'
print('✅ CTR_REFERENCE_DOMICILIATION : normalisation structurée V5 active')


## 5. Contrôles croisés — sans appel Qwen


In [ ]:
CROSS_DOCUMENT_GROUPS=[
    {'name':'SALAIRE_NET','kind':'amount','fields':{
        'ENGAGEMENT_DOMICILIATION':'DOM_SALAIRE_NET_MENSUEL','CONTRAT_TRAVAIL':'CTR_SALAIRE_NET','CONTRAT_SPECIFIQUE':'CTS_SALAIRE_NET'}},
    {'name':'PART_TRANSFERABLE','kind':'amount','fields':{
        'ENGAGEMENT_DOMICILIATION':'DOM_PART_TRANSFERABLE','CONTRAT_SPECIFIQUE':'CTS_PART_TRANSFERABLE'}},
    {'name':'DATE_DEBUT_CONTRAT','kind':'date','fields':{
        'ENGAGEMENT_DOMICILIATION':'DOM_DATE_DEBUT_CONTRAT','CONTRAT_TRAVAIL':'CTR_DATE_DEBUT_CONTRAT','CONTRAT_SPECIFIQUE':'CTS_DATE_DEBUT_CONTRAT'}},
    {'name':'NUMERO_PERMIS','kind':'reference','fields':{
        'CONTRAT_TRAVAIL':'CTR_NUMERO_PERMIS_TRAVAIL','CONTRAT_SPECIFIQUE':'CTS_NUMERO_PERMIS_TRAVAIL','TITRE_TRAVAIL':'TTR_NUMERO_PERMIS'}},
    {'name':'DATE_DEBUT_PERMIS','kind':'date','fields':{
        'CONTRAT_TRAVAIL':'CTR_DATE_DEBUT_VALIDITE_PERMIS','CONTRAT_SPECIFIQUE':'CTS_DATE_DEBUT_VALIDITE_PERMIS','TITRE_TRAVAIL':'TTR_DATE_DEBUT'}},
    {'name':'DATE_FIN_PERMIS','kind':'date','fields':{
        'CONTRAT_TRAVAIL':'CTR_DATE_FIN_VALIDITE_PERMIS','CONTRAT_SPECIFIQUE':'CTS_DATE_FIN_VALIDITE_PERMIS','TITRE_TRAVAIL':'TTR_DATE_FIN'}},
]

def comparable(v): return v not in (None,'')
def same_value(kind,a,b):
    if not comparable(a) or not comparable(b): return True
    if kind=='amount': return abs(float(a)-float(b))<=0.01
    return str(a)==str(b)


In [ ]:

# =====================================================================
# FIELD_CONFIDENCE + DQ_SCORE + APPLICABILITE METIER — V3
# =====================================================================
# IMPORTANT :
# - NORMALIZATION_STATUS décrit UNIQUEMENT la normalisation d'une cellule.
# - DQ_STATUS_FIELD décrit la fiabilité métier du champ après tous les contrôles.
# - VALIDATION_AUTO décrit le statut GLOBAL du dossier.
#
# Un champ peut donc être NORMALIZATION_STATUS=AUTO_OK mais
# DQ_STATUS_FIELD=BLOCKED si sa valeur est en conflit avec un autre document.
#
# Les scores sont des INDICES INTERNES EXPLICABLES, pas des probabilités Qwen.

CRITICAL_FIELDS = {
    'DOM_DATE_DEBUT_CONTRAT','DOM_DATE_FIN_CONTRAT',
    'DOM_SALAIRE_NET_MENSUEL','DOM_PART_TRANSFERABLE',
    'CTR_NUMERO_PERMIS_TRAVAIL','CTR_DATE_DEBUT_VALIDITE_PERMIS',
    'CTR_DATE_FIN_VALIDITE_PERMIS','CTR_SALAIRE_NET',
    'CTS_NUMERO_PERMIS_TRAVAIL','CTS_DATE_DEBUT_VALIDITE_PERMIS',
    'CTS_DATE_FIN_VALIDITE_PERMIS','CTS_SALAIRE_NET',
    'TTR_NUMERO_PERMIS','TTR_DATE_DEBUT','TTR_DATE_FIN',
    'TTR_NOM','TTR_PRENOM'
}

ALWAYS_INFO_ONLY_FIELDS = {
    'CTS_SAP_ID',
    'TTR_NUMERO_MANUSCRIT',
    'PTR_NUMERO_SERIE',
    'PTR_WILAYA',
    'PTR_CACHET_DIRECTION_EMPLOI_PRESENT',
    'CTS_NUMERO_SS_PAYS_ORIGINE',
    'CTS_NUMERO_SS_ALGERIE',
}

NON_PENALIZING_QUALITY_FLAGS = {'MIGRATED_FROM_V8_1'}

DQ_WEIGHTS = {
    'critical_completeness': 25,
    'cross_document': 30,
    'normalization': 15,
    'business_validity': 15,
    'extraction_quality': 10,
    'classification_quality': 5,
}
assert sum(DQ_WEIGHTS.values()) == 100

SEVERITY_RANK = {'OK':0, 'INFO':0, 'REVIEW':1, 'BLOQUANT':2}

DATE_ORDER_FIELDS = {
    'CONTRAT': {'DOM_DATE_DEBUT_CONTRAT','DOM_DATE_FIN_CONTRAT'},
    'PERMIS_CTR': {'CTR_DATE_DEBUT_VALIDITE_PERMIS','CTR_DATE_FIN_VALIDITE_PERMIS'},
    'PERMIS_CTS': {'CTS_DATE_DEBUT_VALIDITE_PERMIS','CTS_DATE_FIN_VALIDITE_PERMIS'},
    'PERMIS_TTR': {'TTR_DATE_DEBUT','TTR_DATE_FIN'},
}

CROSSCHECK_FIELD_MAP = {
    g['name']: set(g['fields'].values())
    for g in CROSS_DOCUMENT_GROUPS
}


def _meaningful(v):
    if v is None:
        return False
    if isinstance(v, str):
        return v.strip().upper() not in NULL_TEXTS
    return True


def _first_value(records, field):
    for r in records:
        v=(r.get('normalized_data') or {}).get(field)
        if comparable(v):
            return v
    return None


def derive_business_context(processed_records):
    cts_records=[r for r in processed_records if r.get('doc_type')=='CONTRAT_SPECIFIQUE']
    has_cts=bool(cts_records)

    old_salary=None
    mention_au_lieu=False
    cts_date_document=None

    for r in cts_records:
        nd=r.get('normalized_data') or {}
        if old_salary is None and comparable(nd.get('CTS_SALAIRE_NET_ANCIEN')):
            old_salary=nd.get('CTS_SALAIRE_NET_ANCIEN')
        if nd.get('CTS_MENTION_AU_LIEU_DE_PRESENTE') is True:
            mention_au_lieu=True
        if cts_date_document is None and comparable(nd.get('CTS_DATE_DOCUMENT')):
            cts_date_document=nd.get('CTS_DATE_DOCUMENT')

    evidence=[]
    if comparable(old_salary):
        evidence.append('CTS_SALAIRE_NET_ANCIEN_PRESENT')
    if mention_au_lieu:
        evidence.append('CTS_MENTION_AU_LIEU_DE_PRESENTE_TRUE')

    if evidence:
        type_dossier='AUGMENTATION'
    elif has_cts:
        type_dossier='NOUVEAU_CONTRAT'
        evidence.append('CTS_SANS_INDICATEUR_AUGMENTATION')
    else:
        type_dossier='A_DETERMINER'
        evidence.append('ABSENCE_CONTRAT_SPECIFIQUE')

    return {
        'TYPE_DOSSIER':type_dossier,
        'TYPE_DOSSIER_MOTIF':' | '.join(evidence),
        # V5 : la date du document CTS reste une information documentaire.
        # Elle NE représente PAS la date d'effet métier de l'augmentation.
        'CTS_DATE_DOCUMENT_AUGMENTATION':cts_date_document if type_dossier=='AUGMENTATION' else None,
        'CTS_DATE_AUGMENTATION':None,
        'PERIODE_EFFET_AUGMENTATION':None,
    }


def field_policy(field, business_context):
    typ=(business_context or {}).get('TYPE_DOSSIER')

    if field in ALWAYS_INFO_ONLY_FIELDS:
        return 'INFO_ONLY','INFO_ONLY'

    if field=='CTS_SALAIRE_NET_ANCIEN' and typ=='NOUVEAU_CONTRAT':
        return 'NON_APPLICABLE','INFO_ONLY'

    if field=='DOM_MONTANT_TOTAL_DOMICILIE' and typ=='AUGMENTATION':
        return 'NON_APPLICABLE','INFO_ONLY'

    if field in CRITICAL_FIELDS:
        return 'APPLICABLE','CRITICAL'

    return 'APPLICABLE','STANDARD'


def is_dq_relevant_field(field, business_context):
    return field_policy(field,business_context)[1] != 'INFO_ONLY'


def is_info_only_document(doc_type, business_context):
    fields=FIELD_SCHEMA.get(doc_type,[])
    return bool(fields) and all(
        not is_dq_relevant_field(f,business_context)
        for f in fields
    )


def relevant_critical_missing(rec, business_context):
    return [
        f for f in (rec.get('critical_fields_missing') or [])
        if is_dq_relevant_field(f,business_context)
    ]


def penalizing_quality_flags(rec):
    return [
        str(x) for x in (rec.get('quality_flags') or [])
        if str(x) not in NON_PENALIZING_QUALITY_FLAGS
    ]


def parse_date_safe(v):
    if not v:
        return None
    try:
        return datetime.strptime(str(v), '%d/%m/%Y')
    except Exception:
        return None


def page_regulatory_flags(rec, business_context=None):
    flags=[]
    info_doc=is_info_only_document(rec.get('doc_type'),business_context)

    if rec.get('classification_review_required') is True and not info_doc:
        flags.append('CLASSIFICATION_REVIEW_REQUIRED')

    status=str(rec.get('extraction_status') or '').upper()
    if not info_doc:
        if status in {'JSON_VIDE','VIDE','FAILED','ECHEC'}:
            flags.append('EXTRACTION_JSON_VIDE')
        elif status in {'PARTIELLE','PARTIAL'}:
            flags.append('EXTRACTION_PARTIELLE')

        if relevant_critical_missing(rec,business_context):
            flags.append('CRITICAL_FIELD_MISSING')

    for qf in penalizing_quality_flags(rec):
        flags.append(qf)

    return list(dict.fromkeys(flags))


def field_evidence(processed_records, field):
    ev=[]
    for r in processed_records:
        nd=r.get('normalized_data') or {}
        tr=r.get('normalization_trace') or {}
        if field in nd:
            ev.append({
                'doc_type':r.get('doc_type'),
                'page':r.get('page_num'),
                'value':nd.get(field),
                'raw':(r.get('raw_data') or {}).get(field),
                'trace':tr.get(field) or {},
                'classification_review_required':bool(r.get('classification_review_required')),
                'quality_flags':penalizing_quality_flags(r),
                'extraction_status':r.get('extraction_status'),
            })
    return ev


def anomaly_applies_to_field(anomaly, field, page=None, doc_type=None):
    """
    Rend cohérents CHAMPS_DETAIL, ANOMALIES et le statut du dossier.
    """
    typ=anomaly.get('TYPE_ANOMALIE')
    achamp=anomaly.get('CHAMP')
    apage=anomaly.get('PAGE')

    if typ=='FORMAT_REVIEW':
        return achamp==field and (apage is None or page==apage)

    if typ in {'CRITICAL_FIELD_MISSING','CRITICAL_FIELD_MISSING_POSTPROCESS'}:
        parts={x.strip() for x in str(achamp or '').split('|') if x.strip()}
        return field in parts

    if typ=='CROSS_DOCUMENT_CONFLICT':
        return field in CROSSCHECK_FIELD_MAP.get(str(achamp),set())

    if typ=='INVALID_DATE_ORDER':
        return field in DATE_ORDER_FIELDS.get(str(achamp),set())

    if typ in {
        'CLASSIFICATION_REVIEW_REQUIRED',
        'EXTRACTION_JSON_VIDE',
        'EXTRACTION_PARTIELLE',
    }:
        return apage is None or page==apage

    return False


def calculate_field_confidence(processed_records, business_context, anomalies):
    rows=[]

    for field in ALL_FIELDS:
        applicability,importance=field_policy(field,business_context)
        ev=field_evidence(processed_records,field)
        present=[e for e in ev if comparable(e['value'])]

        if importance=='INFO_ONLY':
            rows.append({
                'CHAMP':field,
                'APPLICABILITE':applicability,
                'DQ_IMPORTANCE':importance,
                'FIELD_CONFIDENCE':None,
                'FIELD_CONFIDENCE_LEVEL':'INFO_ONLY',
                'FIELD_CONFIDENCE_REASON':'HORS_SCORE_DQ',
                'NB_OCCURRENCES':len(present),
            })
            continue

        if not present:
            score=0
            reasons=['AUCUNE_VALEUR_NORMALISEE']
        else:
            score=70
            reasons=['VALEUR_PRESENTE']

            statuses=[(e['trace'] or {}).get('status') for e in present]
            if any(s=='REVIEW' for s in statuses):
                score-=35
                reasons.append('NORMALISATION_REVIEW')
            elif all(s in {'RAW_OK','AUTO_OK'} for s in statuses):
                score+=10
                reasons.append('NORMALISATION_DETERMINISTE')

            if any(e['classification_review_required'] for e in present):
                score-=30
                reasons.append('CLASSIFICATION_REVIEW')

            if any(str(e['extraction_status'] or '').upper() in
                   {'PARTIELLE','PARTIAL','JSON_VIDE','FAILED','ECHEC'} for e in present):
                score-=20
                reasons.append('EXTRACTION_NON_COMPLETE')

            if any(e['quality_flags'] for e in present):
                score-=5
                reasons.append('QUALITY_FLAG_PRESENT')

            # IMPORTANT V3 :
            # conflit entre champs équivalents de documents différents.
            relevant_conflicts=[
                a for a in anomalies
                if a.get('TYPE_ANOMALIE')=='CROSS_DOCUMENT_CONFLICT'
                and field in CROSSCHECK_FIELD_MAP.get(str(a.get('CHAMP')),set())
            ]
            if relevant_conflicts:
                score-=40
                reasons.append(
                    'CROSS_DOCUMENT_CONFLICT:' +
                    '|'.join(sorted({str(a.get('CHAMP')) for a in relevant_conflicts}))
                )

            if any(
                a.get('TYPE_ANOMALIE')=='INVALID_DATE_ORDER'
                and anomaly_applies_to_field(a,field)
                for a in anomalies
            ):
                score-=40
                reasons.append('INVALID_DATE_ORDER')

            vals=[e['value'] for e in present]
            if len(vals)>=2:
                kind=FIELD_TYPES.get(field,'text')
                if all(same_value(kind,vals[0],x) for x in vals[1:]):
                    score+=20
                    reasons.append(f'CONCORDANCE_{len(vals)}_LECTURES')
                else:
                    score-=35
                    reasons.append('DIVERGENCE_INTER_OCCURRENCES')

            score=max(0,min(100,int(round(score))))

        level='HIGH' if score>=90 else ('MEDIUM' if score>=70 else ('LOW' if score>0 else 'MISSING'))

        rows.append({
            'CHAMP':field,
            'APPLICABILITE':applicability,
            'DQ_IMPORTANCE':importance,
            'FIELD_CONFIDENCE':score,
            'FIELD_CONFIDENCE_LEVEL':level,
            'FIELD_CONFIDENCE_REASON':' | '.join(reasons),
            'NB_OCCURRENCES':len(present),
        })

    return rows


def add_postprocess_critical_missing(processed_records, anomalies, business_context, source):
    """
    Ne dépend pas uniquement du metadata critical_fields_missing de la Partie 1.
    Vérifie directement les valeurs normalisées de la Partie 2.
    """
    existing={
        (a.get('TYPE_ANOMALIE'),str(a.get('CHAMP')))
        for a in anomalies
    }

    for field in CRITICAL_FIELDS:
        if not is_dq_relevant_field(field,business_context):
            continue

        owner=None
        owner_records=[]
        for dt,fields in FIELD_SCHEMA.items():
            if field in fields:
                owner=dt
                owner_records=[r for r in processed_records if r.get('doc_type')==dt]
                break

        # Si le document porteur n'existe pas du tout, on ne transforme pas
        # automatiquement cela en "champ manquant" : cela relève de la
        # complétude documentaire.
        if not owner_records:
            continue

        has_value=any(
            comparable((r.get('normalized_data') or {}).get(field))
            for r in owner_records
        )
        if has_value:
            continue

        key=('CRITICAL_FIELD_MISSING_POSTPROCESS',field)
        if key in existing:
            continue

        anomalies.append({
            'FICHIER':source,
            'PAGE':owner_records[0].get('page_num'),
            'TYPE_DOCUMENT':owner,
            'TYPE_ANOMALIE':'CRITICAL_FIELD_MISSING_POSTPROCESS',
            'SEVERITE':'BLOQUANT',
            'CHAMP':field,
            'VALEUR_RAW':None,
            'VALEUR_NORMALISEE':None,
            'MOTIF':'champ critique absent après normalisation Partie 2'
        })


def add_business_controls(processed, anomalies, business_context):
    records=processed['page_records']

    values={}
    for r in records:
        for f,v in (r.get('normalized_data') or {}).items():
            if comparable(v) and f not in values:
                values[f]=v

    date_pairs=[
        ('DOM_DATE_DEBUT_CONTRAT','DOM_DATE_FIN_CONTRAT','CONTRAT'),
        ('CTR_DATE_DEBUT_VALIDITE_PERMIS','CTR_DATE_FIN_VALIDITE_PERMIS','PERMIS_CTR'),
        ('CTS_DATE_DEBUT_VALIDITE_PERMIS','CTS_DATE_FIN_VALIDITE_PERMIS','PERMIS_CTS'),
        ('TTR_DATE_DEBUT','TTR_DATE_FIN','PERMIS_TTR'),
    ]
    for f1,f2,label in date_pairs:
        d1=parse_date_safe(values.get(f1))
        d2=parse_date_safe(values.get(f2))
        if d1 and d2 and d2<d1:
            anomalies.append({
                'FICHIER':processed['source_file'],
                'PAGE':None,
                'TYPE_DOCUMENT':'MULTI',
                'TYPE_ANOMALIE':'INVALID_DATE_ORDER',
                'SEVERITE':'BLOQUANT',
                'CHAMP':label,
                'VALEUR_RAW':None,
                'VALEUR_NORMALISEE':f'{values.get(f1)} > {values.get(f2)}',
                'MOTIF':'date fin antérieure à date début'
            })

    # V5 : aucune anomalie sur la date d'effet d'augmentation ici.
    # La période d'effet sera fournie par le fichier annuel de paramétrage du planning TL.


def dedupe_anomalies(anomalies):
    seen=set()
    out=[]
    for a in anomalies:
        key=(
            a.get('FICHIER'),a.get('PAGE'),a.get('TYPE_DOCUMENT'),
            a.get('TYPE_ANOMALIE'),a.get('SEVERITE'),a.get('CHAMP'),
            str(a.get('VALEUR_NORMALISEE')),a.get('MOTIF')
        )
        if key not in seen:
            seen.add(key)
            out.append(a)
    return out


def calculate_dq_summary(processed, field_conf_rows):
    records=processed['page_records']
    anomalies=processed.get('anomalies') or []
    business_context=processed.get('business_context') or {}

    field_map={x['CHAMP']:x for x in field_conf_rows}

    critical_scores=[]
    for f in CRITICAL_FIELDS:
        if not is_dq_relevant_field(f,business_context):
            continue
        owner=None
        for dt,fields in FIELD_SCHEMA.items():
            if f in fields:
                owner=dt
                break
        if owner and any(r.get('doc_type')==owner for r in records):
            score=field_map[f].get('FIELD_CONFIDENCE')
            critical_scores.append(1 if score not in (None,0) else 0)

    critical_completeness=(
        sum(critical_scores)/len(critical_scores)
        if critical_scores else 0
    )

    cross_conflicts=[
        a for a in anomalies
        if a.get('TYPE_ANOMALIE')=='CROSS_DOCUMENT_CONFLICT'
    ]
    cross_checks_possible=0
    for g in CROSS_DOCUMENT_GROUPS:
        vals=[]
        for dt,f in g['fields'].items():
            for r in records:
                if r.get('doc_type')==dt:
                    v=(r.get('normalized_data') or {}).get(f)
                    if comparable(v):
                        vals.append(v)
        if len(vals)>=2:
            cross_checks_possible+=1

    cross_document=(
        0.60 if cross_checks_possible==0
        else max(0,1-(len(cross_conflicts)/cross_checks_possible))
    )

    traces=[]
    for r in records:
        for field,tr in (r.get('normalization_trace') or {}).items():
            if is_dq_relevant_field(field,business_context):
                traces.append(tr)
    nonmissing=[t for t in traces if t.get('status')!='MISSING']
    normalization=(
        sum(t.get('status') in {'RAW_OK','AUTO_OK'} for t in nonmissing)/len(nonmissing)
        if nonmissing else 0
    )

    business_bad=sum(
        a.get('TYPE_ANOMALIE') in {'INVALID_DATE_ORDER'}
        for a in anomalies
    )
    business_validity=1.0 if business_bad==0 else 0.0

    page_flags=[page_regulatory_flags(r,business_context) for r in records]
    severe_pages=sum(
        any(x in {'EXTRACTION_JSON_VIDE','EXTRACTION_PARTIELLE','CRITICAL_FIELD_MISSING'} for x in fl)
        for fl in page_flags
    )
    extraction_quality=max(0,1-(severe_pages/max(1,len(records))))

    class_review=sum(
        'CLASSIFICATION_REVIEW_REQUIRED' in fl
        for fl in page_flags
    )
    classification_quality=max(0,1-(class_review/max(1,len(records))))

    components={
        'critical_completeness':critical_completeness,
        'cross_document':cross_document,
        'normalization':normalization,
        'business_validity':business_validity,
        'extraction_quality':extraction_quality,
        'classification_quality':classification_quality,
    }

    dq_score=round(
        sum(components[k]*DQ_WEIGHTS[k] for k in DQ_WEIGHTS),1
    )

    blocking=[a for a in anomalies if a.get('SEVERITE')=='BLOQUANT']
    review=[a for a in anomalies if a.get('SEVERITE')=='REVIEW']

    # V3 : décision cohérente avec l'onglet ANOMALIES.
    # Une anomalie REVIEW empêche AUTO_OK.
    if blocking:
        validation='BLOCKED'
    elif review:
        validation='REVIEW'
    elif dq_score>=90:
        validation='AUTO_OK'
    else:
        validation='REVIEW'

    hard_reasons=list(dict.fromkeys(
        str(a.get('TYPE_ANOMALIE'))
        for a in blocking
    ))

    level='HIGH' if dq_score>=90 else ('MEDIUM' if dq_score>=70 else 'LOW')

    return {
        'TYPE_DOSSIER':business_context.get('TYPE_DOSSIER'),
        'CTS_DATE_DOCUMENT_AUGMENTATION':business_context.get('CTS_DATE_DOCUMENT_AUGMENTATION'),
        'DQ_SCORE':dq_score,
        'DQ_LEVEL':level,
        'VALIDATION_AUTO':validation,
        'HARD_BLOCK':bool(blocking),
        'HARD_BLOCK_REASONS':' | '.join(hard_reasons),
        'NB_BLOQUANTS':len(blocking),
        'NB_REVIEW':len(review),
        'SCORE_CRITICAL_COMPLETENESS':round(critical_completeness*100,1),
        'SCORE_CROSS_DOCUMENT':round(cross_document*100,1),
        'SCORE_NORMALIZATION':round(normalization*100,1),
        'SCORE_BUSINESS_VALIDITY':round(business_validity*100,1),
        'SCORE_EXTRACTION_QUALITY':round(extraction_quality*100,1),
        'SCORE_CLASSIFICATION_QUALITY':round(classification_quality*100,1),
    }


def enrich_field_rows(field_rows, anomalies, field_confidence, dossier_status):
    """
    Ajoute au CHAMPS_DETAIL un statut FINAL cohérent avec les autres feuilles.
    """
    conf_map={x['CHAMP']:x for x in field_confidence}

    for row in field_rows:
        field=row['CHAMP']
        page=row.get('PAGE')
        dt=row.get('TYPE_DOCUMENT')
        importance=row.get('DQ_IMPORTANCE')

        conf=conf_map.get(field,{})
        row['FIELD_CONFIDENCE']=conf.get('FIELD_CONFIDENCE')
        row['FIELD_CONFIDENCE_LEVEL']=conf.get('FIELD_CONFIDENCE_LEVEL')
        row['FIELD_CONFIDENCE_REASON']=conf.get('FIELD_CONFIDENCE_REASON')

        if importance=='INFO_ONLY':
            row['DQ_STATUS_FIELD']='INFO_ONLY'
            row['DQ_SEVERITY']='INFO'
            row['DQ_ANOMALIES']=''
            row['DQ_REASON']='HORS_SCORE_DQ'
            row['DOSSIER_STATUS']=dossier_status
            continue

        related=[
            a for a in anomalies
            if anomaly_applies_to_field(a,field,page,dt)
        ]

        if related:
            maxsev=max(
                (a.get('SEVERITE','REVIEW') for a in related),
                key=lambda x:SEVERITY_RANK.get(x,1)
            )
            row['DQ_STATUS_FIELD']='BLOCKED' if maxsev=='BLOQUANT' else 'REVIEW'
            row['DQ_SEVERITY']=maxsev
            row['DQ_ANOMALIES']=' | '.join(dict.fromkeys(
                str(a.get('TYPE_ANOMALIE')) for a in related
            ))
            row['DQ_REASON']=' | '.join(dict.fromkeys(
                str(a.get('MOTIF')) for a in related if a.get('MOTIF')
            ))
        elif row.get('NORMALIZATION_STATUS')=='MISSING':
            row['DQ_STATUS_FIELD']='MISSING'
            row['DQ_SEVERITY']='INFO'
            row['DQ_ANOMALIES']=''
            row['DQ_REASON']='VALEUR_ABSENTE_NON_BLOQUANTE'
        else:
            row['DQ_STATUS_FIELD']='OK'
            row['DQ_SEVERITY']='OK'
            row['DQ_ANOMALIES']=''
            row['DQ_REASON']='AUCUNE_ANOMALIE_DQ'

        row['DOSSIER_STATUS']=dossier_status

    return field_rows


def build_crosscheck_rows(processed):
    rows=[]
    by_type=defaultdict(list)
    for r in processed['page_records']:
        by_type[r.get('doc_type')].append(r)

    for group in CROSS_DOCUMENT_GROUPS:
        vals=[]
        for dt,field in group['fields'].items():
            for r in by_type.get(dt,[]):
                v=(r.get('normalized_data') or {}).get(field)
                if comparable(v):
                    vals.append({
                        'doc_type':dt,
                        'field':field,
                        'page':r.get('page_num'),
                        'value':v,
                        'raw':(r.get('raw_data') or {}).get(field),
                    })

        if len(vals)<2:
            status='NON_CONTROLE'
            severity='INFO'
        else:
            base=vals[0]['value']
            conflict=any(
                not same_value(group['kind'],base,x['value'])
                for x in vals[1:]
            )
            if conflict:
                status='CONFLIT'
                severity='BLOQUANT' if group['name'] in {
                    'SALAIRE_NET','NUMERO_PERMIS',
                    'DATE_DEBUT_PERMIS','DATE_FIN_PERMIS'
                } else 'REVIEW'
            else:
                status='OK'
                severity='OK'

        rows.append({
            'FICHIER':processed['source_file'],
            'TYPE_DOSSIER':(processed.get('business_context') or {}).get('TYPE_DOSSIER'),
            'CONTROLE':group['name'],
            'STATUS_CONTROLE':status,
            'SEVERITE':severity,
            'NB_VALEURS':len(vals),
            'DETAIL':' | '.join(
                f"{x['doc_type']}.{x['field']}@p{x['page']}={x['value']}"
                for x in vals
            )
        })
    return rows



# =====================================================================
# V4 — PROPOSITION DE VALEUR + CONTROLE MEME CLIENT + PERMIS RTL
# =====================================================================

DOCUMENT_PRIORITY = {
    'CONTRAT_SPECIFIQUE': 1,
    'CONTRAT_TRAVAIL': 2,
    'TITRE_TRAVAIL': 3,
    'ENGAGEMENT_DOMICILIATION': 4,
    'PERMIS_TRAVAIL_COUVERTURE': 99,
}

EQUIVALENT_FIELD_GROUPS = {}
for _g in CROSS_DOCUMENT_GROUPS:
    for _f in _g['fields'].values():
        EQUIVALENT_FIELD_GROUPS[_f] = {
            'name': _g['name'],
            'kind': _g['kind'],
            'fields': dict(_g['fields']),
        }


def _identity_text(v):
    if v in (None, ''):
        return None
    s = str(v).replace('\u00a0', ' ').strip().upper()
    return re.sub(r'[^A-Z0-9]', '', s) or None


def _identity_reference(v):
    if v in (None, ''):
        return None
    s = str(v).replace('\u00a0', ' ').strip().upper()
    s = re.sub(r'\s*([/\-])\s*', r'\1', re.sub(r'\s+', ' ', s)).strip()
    return re.sub(r'[^A-Z0-9]', '', s) or None


def identity_signature(rec):
    dt = rec.get('doc_type')
    nd = rec.get('normalized_data') or {}

    name = dob = permit = None

    if dt == 'CONTRAT_SPECIFIQUE':
        name = nd.get('CTS_NOM_PRENOM_TRAVAILLEUR')
        dob = nd.get('CTS_DATE_NAISSANCE')
        permit = nd.get('CTS_NUMERO_PERMIS_TRAVAIL')

    elif dt == 'CONTRAT_TRAVAIL':
        name = nd.get('CTR_NOM_PRENOM_TRAVAILLEUR')
        dob = nd.get('CTR_DATE_NAISSANCE')
        permit = nd.get('CTR_NUMERO_PERMIS_TRAVAIL')

    elif dt == 'TITRE_TRAVAIL':
        name = ' '.join(
            x for x in [
                str(nd.get('TTR_NOM') or '').strip(),
                str(nd.get('TTR_PRENOM') or '').strip(),
            ] if x
        ) or None
        dob = nd.get('TTR_DATE_NAISSANCE')
        permit = nd.get('TTR_NUMERO_PERMIS')

    elif dt == 'ENGAGEMENT_DOMICILIATION':
        name = nd.get('DOM_NOM_RAISON_SOCIAL_CLIENT')

    return {
        'name': _identity_text(name),
        'dob': str(dob) if comparable(dob) else None,
        'permit': _identity_reference(permit) if comparable(permit) else None,
    }


def compare_identity_signatures(a, b):
    matches, conflicts = [], []

    for key in ('permit', 'dob', 'name'):
        av = (a or {}).get(key)
        bv = (b or {}).get(key)

        if av in (None, '') or bv in (None, ''):
            continue

        if key == 'name':
            ok = (av == bv) or (av in bv) or (bv in av)
        else:
            ok = (av == bv)

        (matches if ok else conflicts).append(key)

    return matches, conflicts


def confirm_same_client(candidate_rec, all_records):
    """
    Confirmation par au moins un AUTRE document :
      - même permis, ou
      - même nom + même date de naissance.
    Une contradiction sur permis/date de naissance empêche la proposition.
    Pour DOM, un nom concordant est accepté comme preuve plus faible.
    """
    cand_sig = identity_signature(candidate_rec)
    confirmations = []
    contradictions = []

    for other in all_records:
        if other is candidate_rec:
            continue
        if other.get('doc_type') == 'PERMIS_TRAVAIL_COUVERTURE':
            continue

        matches, conflicts = compare_identity_signatures(
            cand_sig, identity_signature(other)
        )

        if any(x in {'permit', 'dob'} for x in conflicts):
            contradictions.append(
                f"{other.get('doc_type')}@p{other.get('page_num')}:" +
                ",".join(conflicts)
            )
            continue

        strong = (
            'permit' in matches
            or ('name' in matches and 'dob' in matches)
        )

        if (
            candidate_rec.get('doc_type') == 'ENGAGEMENT_DOMICILIATION'
            and 'name' in matches
            and not conflicts
        ):
            strong = True

        if strong:
            confirmations.append(
                f"{other.get('doc_type')}@p{other.get('page_num')}:" +
                ",".join(matches)
            )

    if contradictions:
        return {
            'status': 'CLIENT_MISMATCH',
            'evidence': ' | '.join(contradictions),
        }

    if confirmations:
        return {
            'status': 'SAME_CLIENT_CONFIRMED',
            'evidence': ' | '.join(confirmations),
        }

    return {
        'status': 'SAME_CLIENT_UNCONFIRMED',
        'evidence': 'Pas assez de preuves croisées',
    }


def candidate_fields_for_target(target_field):
    group = EQUIVALENT_FIELD_GROUPS.get(target_field)
    if not group:
        return []

    items = list(group['fields'].items())
    items.sort(key=lambda x: DOCUMENT_PRIORITY.get(x[0], 99))
    return items


def propose_best_value(processed, target_field, target_page=None, target_doc_type=None):
    """
    Priorité : CTS > CTR > TTR > DOM.
    La cellule en anomalie ne se propose pas elle-même.
    """
    records = processed.get('page_records') or []
    candidates = []

    for dt, field in candidate_fields_for_target(target_field):
        for rec in records:
            if rec.get('doc_type') != dt:
                continue

            value = (rec.get('normalized_data') or {}).get(field)
            tr = (rec.get('normalization_trace') or {}).get(field) or {}

            if not comparable(value) or tr.get('status') == 'REVIEW':
                continue

            same_cell = (
                target_page is not None
                and rec.get('page_num') == target_page
                and target_doc_type == dt
                and field == target_field
            )
            if same_cell:
                continue

            client = confirm_same_client(rec, records)
            if client['status'] != 'SAME_CLIENT_CONFIRMED':
                continue

            candidates.append({
                'priority': DOCUMENT_PRIORITY.get(dt, 99),
                'value': value,
                'source_document': dt,
                'source_page': rec.get('page_num'),
                'source_field': field,
                'same_client_status': client['status'],
                'same_client_evidence': client['evidence'],
            })

    if not candidates:
        return {
            'value': None,
            'source_document': None,
            'source_page': None,
            'source_field': None,
            'same_client_status': 'NO_SAFE_PROPOSAL',
            'same_client_evidence': 'Aucune source confirmée comme même client',
            'rule': 'NO_SAFE_PROPOSAL',
        }

    candidates.sort(key=lambda x: x['priority'])
    best = candidates[0]

    return {
        'value': best['value'],
        'source_document': best['source_document'],
        'source_page': best['source_page'],
        'source_field': best['source_field'],
        'same_client_status': best['same_client_status'],
        'same_client_evidence': best['same_client_evidence'],
        'rule': 'PRIORITY_CTS_CTR_TTR_DOM_WITH_SAME_CLIENT_GATE',
    }


def reverse_reference_group_order(value):
    """
    Inverse uniquement l'ordre des GROUPES, jamais les chiffres d'un groupe.

    000395-26-31/00028138-20
      -> 20-00028138/31-26-000395
    """
    if not comparable(value):
        return None

    s = str(value).strip()
    groups = re.split(r'[/\-]+', s)
    seps = re.findall(r'[/\-]+', s)

    if len(groups) < 2 or len(seps) != len(groups) - 1:
        return None

    groups = list(reversed(groups))
    seps = list(reversed(seps))

    out = groups[0]
    for sep, group in zip(seps, groups[1:]):
        out += sep + group

    return out


def resolve_ttr_permit_rtl(processed_records):
    """
    Corrige TTR_NUMERO_PERMIS seulement si :
      - CTS/CTR donnent le même numéro de permis ;
      - le TTR inversé par GROUPES correspond exactement ;
      - TTR et CTS/CTR ont le même nom + date de naissance.

    Le permis TTR courant est précisément le champ suspect :
    son conflit n'est donc PAS utilisé pour rejeter le test d'identité.
    """
    trusted = []

    for rec in processed_records:
        dt = rec.get('doc_type')
        nd = rec.get('normalized_data') or {}

        if dt == 'CONTRAT_SPECIFIQUE':
            v = nd.get('CTS_NUMERO_PERMIS_TRAVAIL')
        elif dt == 'CONTRAT_TRAVAIL':
            v = nd.get('CTR_NUMERO_PERMIS_TRAVAIL')
        else:
            continue

        if comparable(v):
            trusted.append((dt, rec, v, _identity_reference(v)))

    if not trusted:
        return []

    trusted_keys = {x[3] for x in trusted if x[3]}
    if len(trusted_keys) != 1:
        # CTR/CTS eux-mêmes sont incohérents : ne rien corriger.
        return []

    canonical_key = next(iter(trusted_keys))
    trusted.sort(key=lambda x: DOCUMENT_PRIORITY.get(x[0], 99))
    canonical_value = trusted[0][2]

    corrections = []

    for rec in processed_records:
        if rec.get('doc_type') != 'TITRE_TRAVAIL':
            continue

        nd = rec.get('normalized_data') or {}
        raw = rec.get('raw_data') or {}
        trace = rec.get('normalization_trace') or {}

        current = nd.get('TTR_NUMERO_PERMIS')
        if not comparable(current):
            continue

        if _identity_reference(current) == canonical_key:
            continue

        rtl_candidate = reverse_reference_group_order(current)
        if not rtl_candidate:
            continue

        if _identity_reference(rtl_candidate) != canonical_key:
            continue

        ttr_sig = identity_signature(rec)
        client_ok = False
        evidence = []

        for _, trusted_rec, _, _ in trusted:
            matches, conflicts = compare_identity_signatures(
                ttr_sig, identity_signature(trusted_rec)
            )

            # On IGNORE le conflit sur "permit" car c'est précisément
            # le champ dont on teste l'orientation RTL.
            if 'dob' in conflicts or 'name' in conflicts:
                continue

            if 'name' in matches and 'dob' in matches:
                client_ok = True
                evidence.append(
                    f"{trusted_rec.get('doc_type')}@p{trusted_rec.get('page_num')}:name,dob"
                )

        if not client_ok:
            continue

        previous = current
        nd['TTR_NUMERO_PERMIS'] = canonical_value

        trace['TTR_NUMERO_PERMIS'] = {
            'raw': raw.get('TTR_NUMERO_PERMIS'),
            'normalized': canonical_value,
            'status': 'AUTO_OK',
            'rule': 'TTR_PERMIT_RTL_GROUP_ORDER_RESOLVED_BY_CROSSCHECK',
            'changed': canonical_value != raw.get('TTR_NUMERO_PERMIS'),
            'message': (
                'Ordre des groupes du permis TTR inversé droite->gauche; '
                'résolu par CTR/CTS + concordance nom/date naissance'
            ),
            'previous_normalized': previous,
            'rtl_candidate': rtl_candidate,
            'identity_evidence': ' | '.join(evidence),
        }

        corrections.append({
            'page': rec.get('page_num'),
            'raw': raw.get('TTR_NUMERO_PERMIS'),
            'previous': previous,
            'normalized': canonical_value,
            'rule': 'TTR_PERMIT_RTL_GROUP_ORDER_RESOLVED_BY_CROSSCHECK',
        })

    return corrections



def build_manual_validation_rows(processed, field_rows):
    dq = processed.get('dq_summary') or {}
    ctx = processed.get('business_context') or {}
    out = []

    for r in field_rows:
        if r.get('DQ_STATUS_FIELD') not in {'REVIEW', 'BLOCKED'}:
            continue

        proposal = propose_best_value(
            processed,
            r.get('CHAMP'),
            r.get('PAGE'),
            r.get('TYPE_DOCUMENT'),
        )

        out.append({
            'FICHIER': r.get('FICHIER'),
            'TYPE_DOSSIER': ctx.get('TYPE_DOSSIER'),
            'DOSSIER_STATUS': dq.get('VALIDATION_AUTO'),
            'DQ_SCORE': dq.get('DQ_SCORE'),
            'SEVERITE': r.get('DQ_SEVERITY'),
            'PAGE': r.get('PAGE'),
            'TYPE_DOCUMENT': r.get('TYPE_DOCUMENT'),
            'CHAMP': r.get('CHAMP'),
            'RAW': r.get('RAW'),
            'NORMALIZED': r.get('NORMALIZED'),
            'NORMALIZATION_STATUS': r.get('NORMALIZATION_STATUS'),
            'FIELD_CONFIDENCE': r.get('FIELD_CONFIDENCE'),
            'TYPE_ANOMALIE': r.get('DQ_ANOMALIES'),
            'MOTIF': r.get('DQ_REASON'),

            'VALEUR_PROPOSEE': proposal.get('value'),
            'SOURCE_PROPOSITION_DOCUMENT': proposal.get('source_document'),
            'SOURCE_PROPOSITION_PAGE': proposal.get('source_page'),
            'SOURCE_PROPOSITION_CHAMP': proposal.get('source_field'),
            'MEME_CLIENT_STATUT': proposal.get('same_client_status'),
            'MEME_CLIENT_PREUVE': proposal.get('same_client_evidence'),
            'REGLE_PROPOSITION': proposal.get('rule'),

            'VALEUR_VALIDEE': None,
            'DECISION_MANUELLE': None,
            'COMMENTAIRE': None,
        })

    return out



print('✅ DQ V5 chargé : proposition priorisée + même client + permis RTL + CarthagoDom')


## 6A. V5 — Rapprochement CarthagoDom et validation du numéro de domiciliation

Le rapprochement est volontairement conservateur :

1. `SIEGE_RACINE` est extrait de `DOM_COMPTE_LOCAL` par regex `07310` + 6 chiffres ;
2. le match principal est l'égalité exacte avec `Client` Carthago ;
3. la similarité du nom est **informationnelle uniquement** ;
4. une ligne PREDOM seule n'est jamais proposée comme domiciliation finale ;
5. en cas de plusieurs DOM possibles sans correspondance exacte de référence, aucune valeur n'est imposée.


In [ ]:

import unicodedata
from difflib import SequenceMatcher

try:
    from rapidfuzz import fuzz as _rapidfuzz_fuzz
except Exception:
    _rapidfuzz_fuzz = None


def _text_key(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ''
    s = unicodedata.normalize('NFKD', str(value))
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    s = s.upper().strip()
    s = re.sub(r'[^A-Z0-9]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


def normalize_name_for_match(value):
    return _text_key(value)


def name_similarity(a, b):
    """Score 0..100. Informatif uniquement; jamais utilisé comme clé primaire."""
    a = normalize_name_for_match(a)
    b = normalize_name_for_match(b)
    if not a or not b:
        return None
    if _rapidfuzz_fuzz is not None:
        return round(float(_rapidfuzz_fuzz.token_sort_ratio(a, b)), 1)
    aa = ' '.join(sorted(a.split()))
    bb = ' '.join(sorted(b.split()))
    return round(100.0 * SequenceMatcher(None, aa, bb).ratio(), 1)


def extract_siege_racine(numero_compte):
    """
    Extrait exactement : 07310 + 6 chiffres.
    Exemple 02700731010910800156 -> 07310109108.
    """
    if numero_compte is None:
        return None
    s = str(numero_compte).strip()
    if re.fullmatch(r'\d+\.0+', s):
        s = s.split('.', 1)[0]
    digits = re.sub(r'\D', '', s)
    matches = re.findall(r'07310\d{6}', digits)
    uniq = list(dict.fromkeys(matches))
    return uniq[0] if len(uniq) == 1 else None


def normalize_carthago_client(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    s = str(value).strip()
    if re.fullmatch(r'\d+\.0+', s):
        s = s.split('.', 1)[0]
    digits = re.sub(r'\D', '', s)
    # Excel peut supprimer le zéro initial si Client a été stocké comme nombre.
    if len(digits) == 10 and digits.startswith('7310'):
        digits = '0' + digits
    return digits or None


def _header_key(value):
    return _text_key(value)


def detect_column(columns, exact=None, aliases=()):
    cols = list(columns)
    if exact and exact in cols:
        return exact
    by_key = {_header_key(c): c for c in cols}
    if exact and _header_key(exact) in by_key:
        return by_key[_header_key(exact)]
    for alias in aliases:
        k = _header_key(alias)
        if k in by_key:
            return by_key[k]
    return None


CARTHAGO_DOM_REF_ALIASES = [
    'Numéro de domiciliation', 'Numero de domiciliation', 'N° domiciliation',
    'N° de domiciliation', 'No domiciliation', 'Référence domiciliation',
    'Reference domiciliation', 'Réf domiciliation', 'Ref domiciliation',
    'Numero DOM', 'Numéro DOM', 'N° DOM',
]
CARTHAGO_TYPE_ALIASES = [
    'Type', 'Type dossier', 'Type de dossier', 'Nature dossier',
    'DOM/PREDOM', 'Type DOM/PREDOM', 'Type domiciliation',
]


def classify_carthago_type(value, sheet_name=None):
    txt = _text_key(value)
    sh = _text_key(sheet_name)
    combined = f'{txt} {sh}'.strip()
    if 'PREDOM' in combined or 'PRE DOM' in combined or 'PRE-DOM' in str(value or '').upper():
        return 'PREDOM'
    # N'inférer DOM depuis le nom de feuille que pour des noms explicites.
    if txt:
        if txt == 'DOM' or 'DOMICILIATION' in txt:
            return 'DOM'
    if sh in {'DOM', 'DOMICILIATION', 'DOMICILIATIONS'}:
        return 'DOM'
    return 'UNKNOWN'


def load_carthago_reference(path):
    meta = {
        'available': False,
        'path': str(path),
        'client_col': None,
        'name_col': None,
        'dom_ref_col': None,
        'type_col': None,
        'message': None,
    }
    if not Path(path).exists():
        meta['message'] = 'Fichier CarthagoDom absent; aucune proposition Carthago ne sera faite.'
        return pd.DataFrame(), meta

    xls = pd.ExcelFile(path)
    frames = []
    for sheet in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=sheet, dtype=str)
        if df is None or df.empty:
            continue
        df = df.dropna(how='all').copy()
        if df.empty:
            continue
        df['_SOURCE_SHEET'] = sheet
        frames.append(df)

    if not frames:
        meta['message'] = 'Classeur CarthagoDom sans données exploitables.'
        return pd.DataFrame(), meta

    df = pd.concat(frames, ignore_index=True, sort=False)
    client_col = detect_column(df.columns, CARTHAGO_CLIENT_COL, ['Client'])
    name_col = detect_column(df.columns, CARTHAGO_NAME_COL, ['Nom complet/Raison sociale'])
    dom_ref_col = detect_column(df.columns, CARTHAGO_DOM_REF_COL, CARTHAGO_DOM_REF_ALIASES)
    type_col = detect_column(df.columns, CARTHAGO_TYPE_COL, CARTHAGO_TYPE_ALIASES)

    if not client_col:
        raise ValueError('Colonne Carthago Client introuvable. Renseigner CARTHAGO_CLIENT_COL.')
    if not name_col:
        raise ValueError('Colonne Carthago Nom complet/Raison sociale introuvable. Renseigner CARTHAGO_NAME_COL.')

    df['_CARTHAGO_CLIENT_NORM'] = df[client_col].map(normalize_carthago_client)
    df['_CARTHAGO_NAME_NORM'] = df[name_col].map(normalize_name_for_match)

    if dom_ref_col:
        def _norm_ref(v):
            tr = normalize_ctr_reference_domiciliation_trace(v)
            return tr.get('normalized') if tr.get('status') in {'RAW_OK', 'AUTO_OK'} else None
        df['_CARTHAGO_DOM_REF_NORM'] = df[dom_ref_col].map(_norm_ref)
    else:
        df['_CARTHAGO_DOM_REF_NORM'] = None

    df['_CARTHAGO_TYPE'] = [
        classify_carthago_type(
            row.get(type_col) if type_col else None,
            row.get('_SOURCE_SHEET')
        )
        for _, row in df.iterrows()
    ]
    df['_CARTHAGO_ROW_ID'] = range(1, len(df) + 1)

    meta.update({
        'available': True,
        'client_col': client_col,
        'name_col': name_col,
        'dom_ref_col': dom_ref_col,
        'type_col': type_col,
        'message': 'OK' if dom_ref_col else 'Colonne numéro DOM non détectée; propositions désactivées jusqu’au paramétrage.',
    })
    return df, meta


def _record_value(processed, doc_type, field):
    """Retourne la première valeur normalisée comparable + contexte de page/trace."""
    for rec in processed.get('page_records') or []:
        if rec.get('doc_type') != doc_type:
            continue
        v = (rec.get('normalized_data') or {}).get(field)
        tr = (rec.get('normalization_trace') or {}).get(field) or {}
        raw = (rec.get('raw_data') or {}).get(field)
        if comparable(v) or comparable(raw):
            return {
                'value': v,
                'raw': raw,
                'page': rec.get('page_num'),
                'trace': tr,
            }
    return {'value': None, 'raw': None, 'page': None, 'trace': {}}


def match_domiciliation_carthago(processed, carthago_df, meta):
    compte = _record_value(processed, 'ENGAGEMENT_DOMICILIATION', 'DOM_COMPTE_LOCAL')
    ctr_name = _record_value(processed, 'CONTRAT_TRAVAIL', 'CTR_NOM_PRENOM_TRAVAILLEUR')
    ctr_ref = _record_value(processed, 'CONTRAT_TRAVAIL', 'CTR_REFERENCE_DOMICILIATION')

    siege = extract_siege_racine(compte.get('value') or compte.get('raw'))
    ctr_ref_norm = ctr_ref.get('value')
    ctr_ref_raw = ctr_ref.get('raw')

    summary = {
        'SIEGE_RACINE': siege,
        'DOM_COMPTE_LOCAL': compte.get('value') or compte.get('raw'),
        'CTR_NOM_PRENOM_TRAVAILLEUR': ctr_name.get('value') or ctr_name.get('raw'),
        'CTR_REFERENCE_DOMICILIATION_RAW': ctr_ref_raw,
        'CTR_REFERENCE_DOMICILIATION_NORMALIZED': ctr_ref_norm,
        'CTR_REFERENCE_DOMICILIATION_PAGE': ctr_ref.get('page'),
        'CTR_REFERENCE_DOMICILIATION_NORMALIZATION_STATUS': (ctr_ref.get('trace') or {}).get('status'),
        'CARTHAGO_MATCH_STATUS': None,
        'CARTHAGO_CLIENT': None,
        'CARTHAGO_NOM_COMPLET': None,
        'SIMILARITE_NOM': None,
        'CARTHAGO_TYPE': None,
        'NUMERO_DOMICILIATION_PROPOSE': None,
        'CARTHAGO_CANDIDATS_DOM': None,
        'CARTHAGO_CANDIDATS_PREDOM': None,
        'REGLE_PROPOSITION_DOM': None,
        'CARTHAGO_SOURCE_SHEET': None,
    }
    details = []

    if not meta.get('available'):
        summary['CARTHAGO_MATCH_STATUS'] = 'CARTHAGO_FILE_MISSING'
        summary['REGLE_PROPOSITION_DOM'] = 'NO_PROPOSAL_REFERENCE_FILE_MISSING'
        return summary, details

    if not siege:
        summary['CARTHAGO_MATCH_STATUS'] = 'SIEGE_RACINE_NOT_FOUND'
        summary['REGLE_PROPOSITION_DOM'] = 'NO_PROPOSAL_SIEGE_RACINE_MISSING'
        return summary, details

    candidates = carthago_df[carthago_df['_CARTHAGO_CLIENT_NORM'] == siege].copy()
    if candidates.empty:
        summary['CARTHAGO_MATCH_STATUS'] = 'CLIENT_NOT_FOUND'
        summary['REGLE_PROPOSITION_DOM'] = 'NO_PROPOSAL_CLIENT_NOT_FOUND'
        return summary, details

    client_col = meta['client_col']
    name_col = meta['name_col']
    dom_col = meta['dom_ref_col']

    rows = []
    for _, r in candidates.iterrows():
        sim = name_similarity(
            ctr_name.get('value') or ctr_name.get('raw'),
            r.get(name_col)
        )
        dom_raw = r.get(dom_col) if dom_col else None
        dom_norm = r.get('_CARTHAGO_DOM_REF_NORM') if dom_col else None
        item = {
            'FICHIER': processed.get('source_file'),
            'SIEGE_RACINE': siege,
            'CARTHAGO_ROW_ID': r.get('_CARTHAGO_ROW_ID'),
            'CARTHAGO_SOURCE_SHEET': r.get('_SOURCE_SHEET'),
            'CARTHAGO_CLIENT': r.get(client_col),
            'CTR_NOM_PRENOM_TRAVAILLEUR': ctr_name.get('value') or ctr_name.get('raw'),
            'CARTHAGO_NOM_COMPLET': r.get(name_col),
            'SIMILARITE_NOM': sim,
            'CARTHAGO_TYPE': r.get('_CARTHAGO_TYPE'),
            'CARTHAGO_NUM_DOM_RAW': dom_raw,
            'CARTHAGO_NUM_DOM_NORMALIZED': dom_norm,
            'CTR_REFERENCE_DOMICILIATION_NORMALIZED': ctr_ref_norm,
            'REFERENCE_EXACT_MATCH': bool(ctr_ref_norm and dom_norm and ctr_ref_norm == dom_norm),
        }
        details.append(item)
        rows.append((r, item))

    # Le nom est seulement affiché comme contrôle informationnel.
    best_name_item = max(
        (x[1] for x in rows if x[1].get('SIMILARITE_NOM') is not None),
        key=lambda z: z['SIMILARITE_NOM'],
        default=rows[0][1] if rows else None
    )
    if best_name_item:
        summary['CARTHAGO_CLIENT'] = best_name_item.get('CARTHAGO_CLIENT')
        summary['CARTHAGO_NOM_COMPLET'] = best_name_item.get('CARTHAGO_NOM_COMPLET')
        summary['SIMILARITE_NOM'] = best_name_item.get('SIMILARITE_NOM')

    dom_rows = [x for x in rows if x[1].get('CARTHAGO_TYPE') == 'DOM']
    predom_rows = [x for x in rows if x[1].get('CARTHAGO_TYPE') == 'PREDOM']
    unknown_rows = [x for x in rows if x[1].get('CARTHAGO_TYPE') == 'UNKNOWN']

    def uniq_refs(items):
        vals = []
        for _, item in items:
            v = item.get('CARTHAGO_NUM_DOM_NORMALIZED') or item.get('CARTHAGO_NUM_DOM_RAW')
            if comparable(v) and str(v) not in vals:
                vals.append(str(v))
        return vals

    dom_refs = uniq_refs(dom_rows)
    predom_refs = uniq_refs(predom_rows)
    summary['CARTHAGO_CANDIDATS_DOM'] = ' | '.join(dom_refs) if dom_refs else None
    summary['CARTHAGO_CANDIDATS_PREDOM'] = ' | '.join(predom_refs) if predom_refs else None

    exact_dom = [x for x in dom_rows if x[1].get('REFERENCE_EXACT_MATCH')]
    chosen = None

    if len(exact_dom) == 1:
        chosen = exact_dom[0][1]
        summary['CARTHAGO_MATCH_STATUS'] = 'EXACT_DOM_REFERENCE'
        summary['REGLE_PROPOSITION_DOM'] = 'CLIENT_EXACT + DOM_TYPE + REFERENCE_EXACT'
    elif len(exact_dom) > 1:
        summary['CARTHAGO_MATCH_STATUS'] = 'MULTIPLE_EXACT_DOM_REFERENCE'
        summary['REGLE_PROPOSITION_DOM'] = 'NO_PROPOSAL_MULTIPLE_EXACT_DOM'
    elif len(dom_refs) == 1:
        # Un seul numéro DOM pour ce Client : proposition possible même si CTR est absent/faux.
        chosen = next(item for _, item in dom_rows
                      if str(item.get('CARTHAGO_NUM_DOM_NORMALIZED') or item.get('CARTHAGO_NUM_DOM_RAW')) == dom_refs[0])
        summary['CARTHAGO_MATCH_STATUS'] = 'UNIQUE_DOM_FOR_CLIENT'
        summary['REGLE_PROPOSITION_DOM'] = 'CLIENT_EXACT + UNIQUE_DOM; NAME_SIMILARITY_INFORMATIONAL'
    elif len(dom_refs) > 1:
        summary['CARTHAGO_MATCH_STATUS'] = 'MULTIPLE_DOM_CANDIDATES'
        summary['REGLE_PROPOSITION_DOM'] = 'NO_PROPOSAL_MULTIPLE_DOM_WITHOUT_EXACT_REFERENCE'
    elif predom_refs and not dom_refs:
        summary['CARTHAGO_MATCH_STATUS'] = 'PREDOM_ONLY'
        summary['REGLE_PROPOSITION_DOM'] = 'NO_PROPOSAL_PREDOM_IS_NOT_FINAL_DOM'
    elif unknown_rows:
        summary['CARTHAGO_MATCH_STATUS'] = 'CARTHAGO_TYPE_UNKNOWN'
        summary['REGLE_PROPOSITION_DOM'] = 'NO_PROPOSAL_DOM_PREDOM_TYPE_NOT_IDENTIFIED'
    else:
        summary['CARTHAGO_MATCH_STATUS'] = 'NO_DOM_REFERENCE_AVAILABLE'
        summary['REGLE_PROPOSITION_DOM'] = 'NO_PROPOSAL_DOM_REFERENCE_MISSING'

    if chosen:
        proposed = chosen.get('CARTHAGO_NUM_DOM_NORMALIZED') or chosen.get('CARTHAGO_NUM_DOM_RAW')
        summary['NUMERO_DOMICILIATION_PROPOSE'] = proposed
        summary['CARTHAGO_TYPE'] = chosen.get('CARTHAGO_TYPE')
        summary['CARTHAGO_CLIENT'] = chosen.get('CARTHAGO_CLIENT')
        summary['CARTHAGO_NOM_COMPLET'] = chosen.get('CARTHAGO_NOM_COMPLET')
        summary['SIMILARITE_NOM'] = chosen.get('SIMILARITE_NOM')
        summary['CARTHAGO_SOURCE_SHEET'] = chosen.get('CARTHAGO_SOURCE_SHEET')

    return summary, details


def build_domiciliation_manual_row(processed, match_summary, carthago_meta):
    """
    Une ligne par dossier, même si le numéro OCR semble correct.
    La Partie 2B exigera VALIDEE / CORRIGEE / REJETEE avant certification.
    """
    dq = processed.get('dq_summary') or {}
    ctx = processed.get('business_context') or {}
    proposal = match_summary.get('NUMERO_DOMICILIATION_PROPOSE')
    return {
        'FICHIER': processed.get('source_file'),
        'TYPE_DOSSIER': ctx.get('TYPE_DOSSIER'),
        'DOSSIER_STATUS': dq.get('VALIDATION_AUTO'),
        'DQ_SCORE': dq.get('DQ_SCORE'),
        'SEVERITE': 'CONTROLE_METIER',
        'PAGE': match_summary.get('CTR_REFERENCE_DOMICILIATION_PAGE'),
        'TYPE_DOCUMENT': 'CONTRAT_TRAVAIL',
        'CHAMP': 'NUMERO_DOMICILIATION_VALIDE',
        'RAW': match_summary.get('CTR_REFERENCE_DOMICILIATION_RAW'),
        'NORMALIZED': match_summary.get('CTR_REFERENCE_DOMICILIATION_NORMALIZED'),
        'NORMALIZATION_STATUS': match_summary.get('CTR_REFERENCE_DOMICILIATION_NORMALIZATION_STATUS'),
        'FIELD_CONFIDENCE': None,
        'TYPE_ANOMALIE': 'VALIDATION_NUMERO_DOMICILIATION',
        'MOTIF': match_summary.get('CARTHAGO_MATCH_STATUS'),
        'VALEUR_PROPOSEE': proposal,
        'SOURCE_PROPOSITION_DOCUMENT': 'CARTHAGO_DOM' if proposal else None,
        'SOURCE_PROPOSITION_PAGE': match_summary.get('CARTHAGO_SOURCE_SHEET'),
        'SOURCE_PROPOSITION_CHAMP': carthago_meta.get('dom_ref_col'),
        'MEME_CLIENT_STATUT': 'CLIENT_EXACT' if match_summary.get('CARTHAGO_CLIENT') else match_summary.get('CARTHAGO_MATCH_STATUS'),
        'MEME_CLIENT_PREUVE': (
            f"SIEGE_RACINE={match_summary.get('SIEGE_RACINE')} == Client={match_summary.get('CARTHAGO_CLIENT')}"
            if match_summary.get('CARTHAGO_CLIENT') else None
        ),
        'REGLE_PROPOSITION': match_summary.get('REGLE_PROPOSITION_DOM'),
        'SIEGE_RACINE': match_summary.get('SIEGE_RACINE'),
        'CARTHAGO_CLIENT': match_summary.get('CARTHAGO_CLIENT'),
        'CTR_NOM_PRENOM_TRAVAILLEUR': match_summary.get('CTR_NOM_PRENOM_TRAVAILLEUR'),
        'CARTHAGO_NOM_COMPLET': match_summary.get('CARTHAGO_NOM_COMPLET'),
        'SIMILARITE_NOM': match_summary.get('SIMILARITE_NOM'),
        'CARTHAGO_TYPE': match_summary.get('CARTHAGO_TYPE'),
        'CARTHAGO_MATCH_STATUS': match_summary.get('CARTHAGO_MATCH_STATUS'),
        'CARTHAGO_CANDIDATS_DOM': match_summary.get('CARTHAGO_CANDIDATS_DOM'),
        'CARTHAGO_CANDIDATS_PREDOM': match_summary.get('CARTHAGO_CANDIDATS_PREDOM'),
        'VALEUR_VALIDEE': None,
        'DECISION_MANUELLE': None,
        'COMMENTAIRE': None,
    }


# Tests V5 demandés.
assert extract_siege_racine('02700731010910800156') == '07310109108'
assert name_similarity('YILDIRIM IBRAHIM', 'IBRAHIM YILDIRIM') >= 99
print('✅ SIEGE_RACINE regex : 02700731010910800156 ->', extract_siege_racine('02700731010910800156'))
print('✅ Similarité nom : informationnelle uniquement')


## 6. Traitement d’un JSON RAW


In [ ]:

def validate_raw_contract(d):
    return (
        d.get('schema_version')==SCHEMA_VERSION
        and d.get('field_schema_hash')==EXPECTED_FIELD_SCHEMA_HASH
        and isinstance(d.get('page_records'),list)
    )


def process_raw_dossier(d):
    if not validate_raw_contract(d):
        raise ValueError(f"Contrat RAW incompatible : {d.get('source_file')}")

    source=d['source_file']
    processed_records=[]
    anomalies=[]
    retry=[]

    # --------------------------------------------------------------
    # PASSAGE 1 : NORMALISATION UNIQUEMENT
    # --------------------------------------------------------------
    for rec in d['page_records']:
        dt=rec.get('doc_type')
        raw=dict(rec.get('raw_data') or {})
        normalized={}
        trace={}

        for field in FIELD_SCHEMA.get(dt,[]):
            tr=normalize_field_trace(field,raw.get(field))
            trace[field]=tr
            normalized[field]=tr['normalized']

        p=dict(rec)
        p['normalized_data']=normalized
        p['normalization_trace']=trace
        processed_records.append(p)

    # Résolution sécurisée du cas permis TTR affiché droite->gauche.
    rtl_permit_corrections=resolve_ttr_permit_rtl(processed_records)

    business_context=derive_business_context(processed_records)

    # --------------------------------------------------------------
    # PASSAGE 2 : ANOMALIES LIEES AUX PAGES / NORMALISATION
    # --------------------------------------------------------------
    field_rows=[]

    for rec in processed_records:
        dt=rec.get('doc_type')
        page=rec.get('page_num')
        trace=rec.get('normalization_trace') or {}

        flags=page_regulatory_flags(rec,business_context)
        relevant_missing=relevant_critical_missing(rec,business_context)

        if 'CLASSIFICATION_REVIEW_REQUIRED' in flags:
            anomalies.append({
                'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,
                'TYPE_ANOMALIE':'CLASSIFICATION_REVIEW_REQUIRED',
                'SEVERITE':'BLOQUANT','CHAMP':None,
                'VALEUR_RAW':None,'VALEUR_NORMALISEE':None,
                'MOTIF':'classification page à revoir'
            })

        if 'EXTRACTION_JSON_VIDE' in flags:
            anomalies.append({
                'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,
                'TYPE_ANOMALIE':'EXTRACTION_JSON_VIDE',
                'SEVERITE':'BLOQUANT','CHAMP':None,
                'VALEUR_RAW':None,'VALEUR_NORMALISEE':None,
                'MOTIF':'extraction vide/échouée'
            })

        if 'EXTRACTION_PARTIELLE' in flags:
            anomalies.append({
                'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,
                'TYPE_ANOMALIE':'EXTRACTION_PARTIELLE',
                'SEVERITE':'REVIEW','CHAMP':None,
                'VALEUR_RAW':None,'VALEUR_NORMALISEE':None,
                'MOTIF':'extraction partielle'
            })

        if 'CRITICAL_FIELD_MISSING' in flags:
            anomalies.append({
                'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,
                'TYPE_ANOMALIE':'CRITICAL_FIELD_MISSING',
                'SEVERITE':'BLOQUANT',
                'CHAMP':' | '.join(relevant_missing),
                'VALEUR_RAW':None,'VALEUR_NORMALISEE':None,
                'MOTIF':'champ critique métier manquant (signal Partie 1)'
            })

        for field in FIELD_SCHEMA.get(dt,[]):
            tr=trace[field]
            applicability,importance=field_policy(field,business_context)

            field_rows.append({
                'FICHIER':source,
                'TYPE_DOSSIER':business_context.get('TYPE_DOSSIER'),
                'PAGE':page,
                'TYPE_DOCUMENT':dt,
                'CHAMP':field,
                'TYPE_CHAMP':FIELD_TYPES[field],
                'APPLICABILITE':applicability,
                'DQ_IMPORTANCE':importance,
                'RAW':tr['raw'],
                'NORMALIZED':tr['normalized'],
                # V3 : nom non ambigu.
                'NORMALIZATION_STATUS':tr['status'],
                'NORMALIZATION_RULE':tr['rule'],
                'NORMALIZATION_CHANGED':tr['changed'],
                'NORMALIZATION_MESSAGE':tr.get('message'),
            })

            if tr['status']=='REVIEW' and importance!='INFO_ONLY':
                anomalies.append({
                    'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,
                    'TYPE_ANOMALIE':'FORMAT_REVIEW','SEVERITE':'REVIEW',
                    'CHAMP':field,'VALEUR_RAW':tr['raw'],
                    'VALEUR_NORMALISEE':tr['normalized'],
                    'MOTIF':tr['rule']
                })
                retry.append({
                    'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,
                    'CHAMP':field,'VALEUR_RAW':tr['raw'],
                    'MOTIF':'FORMAT_AMBIGU','RULE':tr['rule']
                })

    # --------------------------------------------------------------
    # PASSAGE 3 : CROSS-CHECK INTER-DOCUMENTS
    # --------------------------------------------------------------
    by_type=defaultdict(list)
    for r in processed_records:
        by_type[r.get('doc_type')].append(r)

    for group in CROSS_DOCUMENT_GROUPS:
        vals=[]
        for dt,field in group['fields'].items():
            for r in by_type.get(dt,[]):
                v=(r.get('normalized_data') or {}).get(field)
                if comparable(v):
                    vals.append((
                        dt,field,r.get('page_num'),v,
                        (r.get('raw_data') or {}).get(field)
                    ))

        if len(vals)>=2:
            base=vals[0][3]
            conflict=any(
                not same_value(group['kind'],base,x[3])
                for x in vals[1:]
            )
            if conflict:
                severity='BLOQUANT' if group['name'] in {
                    'SALAIRE_NET','NUMERO_PERMIS',
                    'DATE_DEBUT_PERMIS','DATE_FIN_PERMIS'
                } else 'REVIEW'

                detail=' | '.join(
                    f'{dt}.{field}@p{p}={v}'
                    for dt,field,p,v,raw in vals
                )

                anomalies.append({
                    'FICHIER':source,'PAGE':None,'TYPE_DOCUMENT':'MULTI',
                    'TYPE_ANOMALIE':'CROSS_DOCUMENT_CONFLICT',
                    'SEVERITE':severity,
                    'CHAMP':group['name'],
                    'VALEUR_RAW':None,
                    'VALEUR_NORMALISEE':detail,
                    'MOTIF':'valeurs divergentes entre documents'
                })

                for dt,field,p,v,raw in vals:
                    retry.append({
                        'FICHIER':source,'PAGE':p,'TYPE_DOCUMENT':dt,
                        'CHAMP':field,'VALEUR_RAW':raw,
                        'MOTIF':'CROSS_DOCUMENT_CONFLICT',
                        'RULE':group['name']
                    })

    # --------------------------------------------------------------
    # PASSAGE 4 : CONTROLES METIER + COMPLETUDE CRITIQUE PARTIE 2
    # --------------------------------------------------------------
    temp_processed={
        'source_file':source,
        'page_records':processed_records,
        'business_context':business_context,
    }

    add_business_controls(temp_processed,anomalies,business_context)

    add_postprocess_critical_missing(
        processed_records,anomalies,business_context,source
    )

    anomalies=dedupe_anomalies(anomalies)

    # --------------------------------------------------------------
    # PASSAGE 5 : CONFIDENCE, SCORE DOSSIER, STATUT FINAL DES CHAMPS
    # --------------------------------------------------------------
    field_confidence=calculate_field_confidence(
        processed_records,business_context,anomalies
    )

    out={
        'schema_version':SCHEMA_VERSION,
        'source_file':source,
        'source_sha256':d.get('source_sha256'),
        'source_pipeline_version':d.get('pipeline_version'),
        'postprocess_version':POSTPROCESS_VERSION,
        'normalization_version':NORMALIZATION_VERSION,
        'processed_at':datetime.now().isoformat(timespec='seconds'),
        'business_context':business_context,
        'page_records':processed_records,
        'anomalies':anomalies,
        'retry_requests':retry,
        'field_confidence':field_confidence,
        'source_stats':d.get('stats') or {},
        'rtl_permit_corrections':rtl_permit_corrections,
    }

    dq_summary=calculate_dq_summary(out,field_confidence)
    out['dq_summary']=dq_summary

    field_rows=enrich_field_rows(
        field_rows,
        anomalies,
        field_confidence,
        dq_summary.get('VALIDATION_AUTO')
    )

    manual_rows=build_manual_validation_rows(out,field_rows)
    crosscheck_rows=build_crosscheck_rows(out)

    return (
        out,field_rows,anomalies,retry,field_confidence,
        dq_summary,manual_rows,crosscheck_rows
    )


def first_non_null(*values):
    for v in values:
        if v not in (None,''):
            return v
    return None


def consolidate_for_validation(processed):
    records=processed['page_records']
    ctx=processed.get('business_context') or {}

    row={
        'FICHIER':processed['source_file'],
        'TYPE_DOSSIER':ctx.get('TYPE_DOSSIER'),
        'TYPE_DOSSIER_MOTIF':ctx.get('TYPE_DOSSIER_MOTIF'),
        'CTS_DATE_DOCUMENT_AUGMENTATION':ctx.get('CTS_DATE_DOCUMENT_AUGMENTATION'),
        'PERIODE_EFFET_AUGMENTATION':ctx.get('PERIODE_EFFET_AUGMENTATION'),
    }

    row['NB_PAGES']=len(set(
        r.get('page_num') for r in records
        if r.get('page_num') is not None
    ))

    row['TYPES_DOCUMENTS']=' | '.join(dict.fromkeys(
        str(r.get('doc_type')) for r in records
    ))

    for r in records:
        nd=r.get('normalized_data') or {}
        for f,v in nd.items():
            if f not in row or row.get(f) in (None,''):
                row[f]=v

    for f in ALL_FIELDS:
        row.setdefault(f,None)

    row['NOM_TRAVAILLEUR_REFERENCE']=first_non_null(
        row.get('CTR_NOM_PRENOM_TRAVAILLEUR'),
        row.get('CTS_NOM_PRENOM_TRAVAILLEUR'),
        ' '.join(
            x for x in [
                str(row.get('TTR_NOM') or '').strip(),
                str(row.get('TTR_PRENOM') or '').strip()
            ] if x
        ) or None
    )

    row['NUMERO_PERMIS_REFERENCE']=first_non_null(
        row.get('TTR_NUMERO_PERMIS'),
        row.get('CTR_NUMERO_PERMIS_TRAVAIL'),
        row.get('CTS_NUMERO_PERMIS_TRAVAIL')
    )

    row['DATE_DEBUT_CONTRAT_REFERENCE']=first_non_null(
        row.get('CTS_DATE_DEBUT_CONTRAT'),
        row.get('CTR_DATE_DEBUT_CONTRAT'),
        row.get('DOM_DATE_DEBUT_CONTRAT')
    )

    row['DATE_FIN_CONTRAT_REFERENCE']=first_non_null(
        row.get('DOM_DATE_FIN_CONTRAT')
    )

    row['SALAIRE_REFERENCE']=first_non_null(
        row.get('CTS_SALAIRE_NET'),
        row.get('CTR_SALAIRE_NET'),
        row.get('DOM_SALAIRE_NET_MENSUEL')
    )

    row['PART_TRANSFERABLE_REFERENCE']=first_non_null(
        row.get('CTS_PART_TRANSFERABLE'),
        row.get('DOM_PART_TRANSFERABLE')
    )

    # Références vigilance — priorité CTS.
    row['PERE_NOM_PRENOM_REFERENCE']=first_non_null(
        row.get('CTS_PERE_NOM_PRENOM'),
        row.get('CTR_PERE_NOM_PRENOM')
    )
    row['MERE_NOM_PRENOM_REFERENCE']=first_non_null(
        row.get('CTS_MERE_NOM_PRENOM'),
        row.get('CTR_MERE_NOM_PRENOM')
    )
    row['DUREE_CONTRAT_MOIS_REFERENCE']=first_non_null(
        row.get('CTS_DUREE_MOIS'),
        row.get('CTR_DUREE_MOIS'),
        row.get('DOM_DUREE_CONTRAT_MOIS')
    )

    anomalies=processed.get('anomalies') or []
    row['NB_ANOMALIES']=len(anomalies)
    row['NB_BLOQUANTS']=sum(a.get('SEVERITE')=='BLOQUANT' for a in anomalies)
    row['NB_REVIEW']=sum(a.get('SEVERITE')=='REVIEW' for a in anomalies)

    dq=processed.get('dq_summary') or {}
    for k,v in dq.items():
        if k not in {'TYPE_DOSSIER','CTS_DATE_AUGMENTATION','CTS_DATE_DOCUMENT_AUGMENTATION','PERIODE_EFFET_AUGMENTATION'}:
            row[k]=v

    row['NB_RETRY_REQUESTS']=len(processed.get('retry_requests') or [])
    row['STATUT_VALIDATION']=row.get('VALIDATION_AUTO','REVIEW')
    row['COMMENTAIRE_VALIDATION']=None

    return row,{}


## 7. Exécution Partie 2 et classeur de validation


In [ ]:


# ---------------------------------------------------------------------
# V5 — Chargement CarthagoDom
# ---------------------------------------------------------------------
CARTHAGO_DF, CARTHAGO_META = load_carthago_reference(CARTHAGO_XLSX)
print('CarthagoDom :', CARTHAGO_META.get('message'))
print('  Client    :', CARTHAGO_META.get('client_col'))
print('  Nom       :', CARTHAGO_META.get('name_col'))
print('  Num DOM   :', CARTHAGO_META.get('dom_ref_col'))
print('  Type      :', CARTHAGO_META.get('type_col'))
if CARTHAGO_META.get('available'):
    print('  Lignes    :', len(CARTHAGO_DF))

json_files=sorted(RAW_JSON_DIR.glob('*.json'))
if MAX_DOSSIERS is not None:
    json_files=json_files[:int(MAX_DOSSIERS)]
print('JSON RAW sélectionnés :',len(json_files))

all_dossiers=[]
all_fields=[]
all_anomalies=[]
all_retry=[]
all_docs=[]
all_field_confidence=[]
all_dq_summary=[]
all_manual=[]
all_crosschecks=[]
all_carthago_matches=[]
errors=[]

for i,p in enumerate(json_files,1):
    try:
        raw=json.loads(p.read_text(encoding='utf-8'))

        (
            processed,field_rows,anomalies,retry,
            field_confidence,dq_summary,
            manual_rows,crosscheck_rows
        )=process_raw_dossier(raw)

        # V5 — rapprochement Carthago et ligne de validation du numéro DOM.
        carthago_summary, carthago_details = match_domiciliation_carthago(
            processed, CARTHAGO_DF, CARTHAGO_META
        )
        processed['domiciliation_validation'] = carthago_summary
        manual_rows.append(
            build_domiciliation_manual_row(processed, carthago_summary, CARTHAGO_META)
        )
        all_carthago_matches.extend(carthago_details)

        row,_=consolidate_for_validation(processed)
        row.update({
            'SIEGE_RACINE': carthago_summary.get('SIEGE_RACINE'),
            'NUMERO_DOMICILIATION_OCR': carthago_summary.get('CTR_REFERENCE_DOMICILIATION_NORMALIZED'),
            'NUMERO_DOMICILIATION_PROPOSE': carthago_summary.get('NUMERO_DOMICILIATION_PROPOSE'),
            'CARTHAGO_MATCH_STATUS': carthago_summary.get('CARTHAGO_MATCH_STATUS'),
            'SIMILARITE_NOM_CARTHAGO': carthago_summary.get('SIMILARITE_NOM'),
        })

        all_dossiers.append(row)
        all_fields.extend(field_rows)
        all_anomalies.extend(anomalies)
        all_retry.extend(retry)
        all_manual.extend(manual_rows)
        all_crosschecks.extend(crosscheck_rows)

        all_field_confidence.extend([
            {'FICHIER':processed['source_file'], **x}
            for x in field_confidence
        ])

        all_dq_summary.append({
            'FICHIER':processed['source_file'],
            'TYPE_DOSSIER':(processed.get('business_context') or {}).get('TYPE_DOSSIER'),
            'CTS_DATE_DOCUMENT_AUGMENTATION':(processed.get('business_context') or {}).get('CTS_DATE_DOCUMENT_AUGMENTATION'),
            **dq_summary,
            'NB_ANOMALIES':len(anomalies)
        })

        for r in processed['page_records']:
            all_docs.append({
                'FICHIER':processed['source_file'],
                'PAGE':r.get('page_num'),
                'TYPE_DOCUMENT':r.get('doc_type'),
                'STATUT_EXTRACTION':r.get('extraction_status'),
                'TAUX_REMPLISSAGE':r.get('extraction_taux_remplissage'),
                'CLASSIFICATION_REVIEW_REQUIRED':r.get('classification_review_required'),
                'CRITICAL_FIELDS_MISSING_RAW':' | '.join(r.get('critical_fields_missing') or []),
                'CRITICAL_FIELDS_MISSING_DQ':' | '.join(
                    relevant_critical_missing(
                        r,processed.get('business_context') or {}
                    )
                ),
                'QUALITY_FLAGS':' | '.join(r.get('quality_flags') or []),
                'STRATEGIES':' > '.join(
                    s.get('nom','')
                    for s in (r.get('extraction_strategies') or [])
                ),
                'TOKENS_IN':r.get('extraction_tokens_in'),
                'TOKENS_OUT':r.get('extraction_tokens_out')
            })

        outpath=PROCESSED_JSON_DIR/p.name
        outpath.write_text(
            json.dumps(processed,ensure_ascii=False,indent=2,default=str),
            encoding='utf-8'
        )

        print(
            f'[{i}/{len(json_files)}] ✅ {p.name} | '
            f"status={dq_summary.get('VALIDATION_AUTO')} | "
            f"anomalies={len(anomalies)}"
        )

    except Exception as exc:
        errors.append({'FICHIER':p.name,'ERREUR':repr(exc)})
        print(f'[{i}/{len(json_files)}] ❌ {p.name}: {exc}')


DF_DOSSIERS=pd.DataFrame(all_dossiers)
DF_FIELDS=pd.DataFrame(all_fields)
DF_ANOMALIES=pd.DataFrame(all_anomalies)
DF_RETRY=(
    pd.DataFrame(all_retry).drop_duplicates()
    if all_retry else
    pd.DataFrame(columns=[
        'FICHIER','PAGE','TYPE_DOCUMENT','CHAMP',
        'VALEUR_RAW','MOTIF','RULE'
    ])
)
DF_DOCS=pd.DataFrame(all_docs)
DF_ERRORS=pd.DataFrame(errors)
DF_FIELD_CONFIDENCE=pd.DataFrame(all_field_confidence)
DF_DQ_SUMMARY=pd.DataFrame(all_dq_summary)
DF_MANUAL=pd.DataFrame(all_manual)
DF_CROSSCHECK=pd.DataFrame(all_crosschecks)
DF_CARTHAGO_MATCH=pd.DataFrame(all_carthago_matches)

DF_DOSSIERS.to_csv(DOSSIERS_CSV,index=False,encoding='utf-8-sig')
DF_FIELDS.to_csv(FIELDS_CSV,index=False,encoding='utf-8-sig')
DF_ANOMALIES.to_csv(ANOMALIES_CSV,index=False,encoding='utf-8-sig')
DF_RETRY.to_csv(RETRY_CSV,index=False,encoding='utf-8-sig')
DF_FIELD_CONFIDENCE.to_csv(FIELD_CONFIDENCE_CSV,index=False,encoding='utf-8-sig')
DF_DQ_SUMMARY.to_csv(DQ_SUMMARY_CSV,index=False,encoding='utf-8-sig')
DF_CARTHAGO_MATCH.to_csv(CARTHAGO_MATCH_CSV,index=False,encoding='utf-8-sig')

with pd.ExcelWriter(VALIDATION_XLSX,engine='openpyxl') as writer:
    # Ordre des feuilles : opérationnel d'abord.
    DF_DQ_SUMMARY.to_excel(writer,sheet_name='DQ_DASHBOARD',index=False)
    DF_MANUAL.to_excel(writer,sheet_name='A_VALIDER_MANUELLEMENT',index=False)
    DF_DOSSIERS.to_excel(writer,sheet_name='DOSSIERS_A_VALIDER',index=False)
    DF_CROSSCHECK.to_excel(writer,sheet_name='CONTROLES_COHERENCE',index=False)
    DF_CARTHAGO_MATCH.to_excel(writer,sheet_name='CARTHAGO_MATCH',index=False)
    DF_FIELDS.to_excel(writer,sheet_name='CHAMPS_DETAIL',index=False)
    DF_ANOMALIES.to_excel(writer,sheet_name='ANOMALIES',index=False)
    DF_FIELD_CONFIDENCE.to_excel(writer,sheet_name='FIELD_CONFIDENCE',index=False)
    DF_RETRY.to_excel(writer,sheet_name='VLM_RETRY_REQUESTS',index=False)
    DF_DOCS.to_excel(writer,sheet_name='DOCUMENTS',index=False)
    DF_ERRORS.to_excel(writer,sheet_name='ERREURS',index=False)

    pd.DataFrame([
        {'PARAMETRE':'schema_version','VALEUR':SCHEMA_VERSION},
        {'PARAMETRE':'field_schema_hash','VALEUR':EXPECTED_FIELD_SCHEMA_HASH},
        {'PARAMETRE':'postprocess_version','VALEUR':POSTPROCESS_VERSION},
        {'PARAMETRE':'normalization_version','VALEUR':NORMALIZATION_VERSION},
        {'PARAMETRE':'normalization_status_note','VALEUR':'NORMALIZATION_STATUS concerne uniquement le format de la cellule'},
        {'PARAMETRE':'field_status_note','VALEUR':'DQ_STATUS_FIELD intègre normalisation + extraction + cross-check + contrôles métier'},
        {'PARAMETRE':'dossier_status_note','VALEUR':'AUTO_OK uniquement si aucune anomalie BLOQUANT/REVIEW et DQ_SCORE >= 90'},
        {'PARAMETRE':'dq_score_note','VALEUR':'Indice interne explicable 0-100; ce n est pas une probabilité ni un taux de confiance Qwen'},
        {'PARAMETRE':'qwen_confidence','VALEUR':'NON INTEGRE - chantier pré-production séparé'},
        {'PARAMETRE':'type_dossier_rule','VALEUR':'AUGMENTATION si CTS_SALAIRE_NET_ANCIEN présent ou CTS_MENTION_AU_LIEU_DE_PRESENTE=True; sinon NOUVEAU_CONTRAT si CTS présent'},
        {'PARAMETRE':'augmentation_effect_rule','VALEUR':'V5: la date CTS/document n est PAS la date d effet; le planning utilisera un fichier annuel ANNEE_AUGMENTATION + MOIS_AUGMENTATION'},
        {'PARAMETRE':'manual_proposal_priority','VALEUR':'CONTRAT_SPECIFIQUE > CONTRAT_TRAVAIL > TITRE_TRAVAIL > ENGAGEMENT_DOMICILIATION'},
        {'PARAMETRE':'manual_proposal_same_client','VALEUR':'Proposition uniquement si le document source est confirmé comme même client'},
        {'PARAMETRE':'permit_rtl_rule','VALEUR':'TTR permis: inversion ordre des groupes uniquement si match CTR/CTS + identité nom/date naissance'},
        {'PARAMETRE':'ctr_dom_ref_rule','VALEUR':'Format canonique PREFIXE|AAAA.T|40|SEQUENCE|DZD; zéros initiaux conservés; aucune correction OCR ambiguë'},
        {'PARAMETRE':'siege_racine_rule','VALEUR':'Regex 07310 + 6 chiffres depuis DOM_COMPTE_LOCAL; aucune découpe par position fixe'},
        {'PARAMETRE':'carthago_match_rule','VALEUR':'SIEGE_RACINE == Client; similarité nom informationnelle; PREDOM seul jamais proposé comme DOM final'},
    ]).to_excel(writer,sheet_name='PARAMETRES',index=False)

    # Format monétaire 2 décimales dans le consolidé.
    ws=writer.book['DOSSIERS_A_VALIDER']
    amount_cols=set(AMOUNT_FIELDS) | {
        'SALAIRE_REFERENCE','PART_TRANSFERABLE_REFERENCE'
    }
    headers={cell.value:cell.column for cell in ws[1]}
    for col_name in amount_cols:
        col_idx=headers.get(col_name)
        if col_idx:
            for row_idx in range(2,ws.max_row+1):
                cell=ws.cell(row=row_idx,column=col_idx)
                if isinstance(cell.value,(int,float)):
                    cell.number_format='0.00'

    # Format montant CHAMPS_DETAIL.
    ws=writer.book['CHAMPS_DETAIL']
    headers={cell.value:cell.column for cell in ws[1]}
    type_col=headers.get('TYPE_CHAMP')
    norm_col=headers.get('NORMALIZED')
    if type_col and norm_col:
        for row_idx in range(2,ws.max_row+1):
            if ws.cell(row=row_idx,column=type_col).value=='amount':
                cell=ws.cell(row=row_idx,column=norm_col)
                if isinstance(cell.value,(int,float)):
                    cell.number_format='0.00'


# Liste déroulante DECISION_MANUELLE.
if 'A_VALIDER_MANUELLEMENT' in writer.book.sheetnames:
    from openpyxl.worksheet.datavalidation import DataValidation

    ws_manual = writer.book['A_VALIDER_MANUELLEMENT']
    manual_headers = {
        cell.value: cell.column
        for cell in ws_manual[1]
    }
    decision_col = manual_headers.get('DECISION_MANUELLE')

    if decision_col and ws_manual.max_row >= 2:
        dv = DataValidation(
            type='list',
            formula1='"VALIDEE,CORRIGEE,REJETEE"',
            allow_blank=True
        )
        dv.error = 'Choisir VALIDEE, CORRIGEE ou REJETEE.'
        dv.errorTitle = 'Décision invalide'
        dv.prompt = 'Sélectionner la décision de validation.'
        dv.promptTitle = 'Décision manuelle'
        ws_manual.add_data_validation(dv)

        start_cell = ws_manual.cell(row=2, column=decision_col).coordinate
        end_cell = ws_manual.cell(row=ws_manual.max_row, column=decision_col).coordinate
        dv.add(f'{start_cell}:{end_cell}')

    # Figer la première ligne sur les feuilles les plus utilisées.
    for sheet_name in [
        'DQ_DASHBOARD','A_VALIDER_MANUELLEMENT',
        'DOSSIERS_A_VALIDER','CONTROLES_COHERENCE',
        'CHAMPS_DETAIL','ANOMALIES','CARTHAGO_MATCH'
    ]:
        writer.book[sheet_name].freeze_panes='A2'

    # Les validations/freeze panes sont ajoutés après la fermeture du writer pandas.
    # Sauvegarde explicite pour garantir leur persistance dans le fichier final.
    writer.book.save(VALIDATION_XLSX)

print('\n✅ Classeur validation :',VALIDATION_XLSX)
print('✅ Dossiers CSV       :',DOSSIERS_CSV)
print('✅ Champs détail      :',FIELDS_CSV)
print('✅ Anomalies          :',ANOMALIES_CSV)
print('✅ Retry requests     :',RETRY_CSV)
print('✅ DQ summary         :',DQ_SUMMARY_CSV)
print('✅ Field confidence   :',FIELD_CONFIDENCE_CSV)
print('✅ Carthago match      :',CARTHAGO_MATCH_CSV)
print('✅ Nouvelle feuille   : A_VALIDER_MANUELLEMENT')
print('✅ Nouvelle feuille   : CONTROLES_COHERENCE')
print('✅ Nouvelle feuille   : CARTHAGO_MATCH')
print('✅ Validation DOM     : NUMERO_DOMICILIATION_VALIDE ajouté à A_VALIDER_MANUELLEMENT')


## 8. Étape suivante après validation V5

Dans `A_VALIDER_MANUELLEMENT`, traiter les anomalies DQ **et** la ligne `NUMERO_DOMICILIATION_VALIDE` de chaque dossier avec :

- `VALIDEE` : conserver la valeur proposée/normalisée selon la règle Partie 2B ;
- `CORRIGEE` : renseigner obligatoirement `VALEUR_VALIDEE` ;
- `REJETEE` : la domiciliation n'est pas certifiée (par exemple DOM absente de CarthagoDom).

La Partie 2B produira ensuite les données certifiées. Le futur `PLANNING_TL` utilisera uniquement le numéro de domiciliation validé, jamais directement la valeur OCR.
